## 1. Load expert and amateur models

### Optional: Uninstall the Dataset

If the dataset has already been loaded, you may need to uninstall it to avoid conflicts.  
**Remove the `#` symbol below to run the command.**

```python
!pip uninstall datasets
```

In [ ]:
!pip install transformers
# !pip uninstall datasets
!pip install datasets
!pip install --upgrade transformers

Found existing installation: datasets 2.14.4
Uninstalling datasets-2.14.4:
  Would remove:
    /usr/local/bin/datasets-cli
    /usr/local/lib/python3.11/dist-packages/datasets-2.14.4.dist-info/*
    /usr/local/lib/python3.11/dist-packages/datasets/*
Proceed (Y/n)? y
  Successfully uninstalled datasets-2.14.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_m

## Login HuggingFace


In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) 

Or logging in by this method

In [ ]:
from huggingface_hub import login

login(token="hf_xxxxxxxxxxxxx")  # Replace with your actual token

## General Data Loading Setup

**Purpose**:
Loads a a specified Hugging Face dataset and returns the full training and testing sets along with a small randomized subset (debug set) for quick testing or inspection.
**Parameters**:
* `dataset_name`: name of the dataset to load from the HuggingFace `datasets` library
* `if_gsm8k`: If `True`, uses the `main` configuration. If `False`, loads the dataset using its default configuration


**Returns**:
* `train_dataset`: The full training split
* `test_dataset`: the full test split
* `debug_dataset`: a small subset (5 samples) randomly selected from the training dataset for debugging or inspection purposes

In [ ]:
import random
import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader

# Set seeds for reproducibility
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

def generate_dataset(dataset_name, if_gsm8k = False):
    # Load dataset splits
    if if_gsm8k:
      train_dataset = load_dataset(dataset_name, 'main', split="train")
      test_dataset = load_dataset(dataset_name, 'main', split="test")
    else:
      train_dataset = load_dataset(dataset_name, split="train")
      test_dataset = load_dataset(dataset_name, split="test")

    # Generate 5 random indices from the training set
    rand_indices = [random.randint(0, len(train_dataset) - 1) for _ in range(5)]
    debug_dataset = train_dataset.select(rand_indices)

    return train_dataset, test_dataset, debug_dataset

**Purpose**:
Wraps the given datasets into PyTorch objects for batched iterations during training, evaluation, or debugging

**Parameters**:
* `train_dataset`, `test_dataset`, `debug_dataset`: Datasets previously loaded using `generate_dataset`

**Returns**:
* `train_dataloader`: `DataLoader` object for the training dataset
* `test_dataloader`: `DataLoader` for the test set
* `debug_dataloader`: `DataLoader` for the debug dataset

**Note**:
The batch size is currently set to `1` (for debugging). For real training, it's recommmended to increase to powers of 2 like 16, 32, or 64

In [ ]:
## For training purposes, we should increase the batch size to values like 16, 32, or 64 — typically powers of 2. For now, batch size = 1 for debug

batch_size = 1
def generate_dataloader(train_dataset, test_dataset, debug_dataset):
  train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle = False)
  test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle = False)
  debug_dataloader = DataLoader(debug_dataset, batch_size=batch_size, shuffle = False)

  return (train_dataloader, test_dataloader, debug_dataloader)

**Purpose**:
Prints the first examples of two selected features from a Hugging Face dataset. This is useful for quickly inspecting dataset structure or confirming data cleaning
**Parameters**:
* `data`: A Hugging Face dataset (e.g. train/test/debug)
* `feature1`: The first feature key to print (e.g., `question`, `prompt`)
* `feature2`; The seconf feature key to print (e.g., `answwer`)

In [ ]:
def print_first_features_from_dataset(data, feature1, feature2):
    for i, item in enumerate(data):
        if i == 0:
            first_feature = data[feature1][0]
            second_feature = data[feature2][0]
            print(first_feature)
            print(second_feature)

**Purpose**:
Cleans all `answer` fields in GMS8K like dataset by removing metadata tokens of the for `<<...>>` using regular expressions. This helps simplify the text for training or evaluation.
**Returns**:
* Cleaned versins of `train_dataset`, `test_dataset`, and `debug_dataset`, where metadata tags are removed from the `"answer"` fields.

**How it works**:
* Use regex substitution (`re.sub`) to remove any text enclosed in `<<` and `>>`

In [ ]:
def gsm8k_clean_data(train_dataset, test_dataset, debug_dataset):
    import re

    def clean_data(text):
        return re.sub(r'<<.*?>>', '', text)

    train_dataset = train_dataset.map(lambda x: {"answer": clean_data(x["answer"])})
    test_dataset = test_dataset.map(lambda x: {"answer": clean_data(x["answer"])})
    debug_dataset = debug_dataset.map(lambda x: {"answer": clean_data(x["answer"])})

    return train_dataset, test_dataset, debug_dataset

### Loading Data and Models

This "Save Processed Data" section only needs to be run once.
It sets up the expert, amateur, and base models along with the relevant datasets,
and saves them to Google Drive for future use.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_list = ["gsm8k", "commen_gen", "whatsthatbook"]
dataset_id = ["openai/gsm8k", "allenai/common_gen", "nlpkevinl/whatsthatbook"]

# Add expert into the list later
models_list = ["amateur", "base"]
checkpoint_list = ["meta-llama/Llama-2-7b-hf", "microsoft/phi-1_5"]

**Purpose for function `loading_dataset`**:

This function is responsible for loading and preparing multiple datasets. For each dataset, it generates three subsets: training, testing, and debugging. If the dataset is `gsm8k`, it also performs answer-cleaning to remove unwanted tokens. The dataset are loaded from the Hugging Face `datasets` library and then processed into their respective DataLoaders for use in training and evaluation.

**Parameters**:
* `dataset_list`: A list of dataset names as strings.
* `dataset_id_list`: A list of corresponding Hugging Face dataset IDs. The indices must match `dataset_list`.

**Returns**:
A list of tuples, each containing `(train_dataset, test_dataset, debug_dataset)` for each dataset. These can be used directly for training, evaluation, and debuggin purposes.


**Puropse for function `loading_model`:

This function loads pretrained causal language models and their corresponding tokenizers from given checkpoint paths using Hugging Face's `transformers` library. It allows multiple models in one go, pairing each with a descriptive model name.

**Parameters**:
* `model_name_list`: a list of human-readable names to identify each model
* `checkpoint_path_list`: a list of hugging face model checkpoint paths that point to pretrained models.





In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

def loading_dataset(dataset_list, dataset_id_list):
    all_datasets = []

    for i in range(len(dataset_list)):
        dataset_name = dataset_list[i]
        dataset_id = dataset_id_list[i]
        is_gsm8k = dataset_name.lower() == "gsm8k"

        # Generate datasets
        train_dataset, test_dataset, debug_dataset = generate_dataset(dataset_id, is_gsm8k)

        if is_gsm8k:
            train_dataset, test_dataset, debug_dataset = gsm8k_clean_data(train_dataset, test_dataset, debug_dataset)
        # Generate dataloaders
        train_dataloader, test_dataloader, debug_dataloader = generate_dataloader(
            train_dataset, test_dataset, debug_dataset
        )

        # Optionally collect datasets for further use
        all_datasets.append((train_dataset, test_dataset, debug_dataset))

    return all_datasets

def loading_model(model_name_list, checkpoint_path_list):
    """
    Load models and tokenizers from checkpoints and save them under model_name_list.
    """

    models_and_tokenizers = []

    for i in range(len(model_name_list)):
        checkpoint_path = checkpoint_path_list[i]
        model_name = model_name_list[i]

        tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

        models_and_tokenizers.append({
          "model_name" : model_name_list[i],
          "model" : model,
          "tokenizer" : tokenizer
        })

    return models_and_tokenizers

### Save/Reload Data

**Purpose**:
Appends a dictionary of results to a CSV file, storing only the specified features (columns) in order.

**Parameters**:
* `result`: A dictionary containing the data you want to save (e.g., `"answer": "Yes", "score": 0.9}`)
* `features`: a list of feature names you expect in the result. The function writes values in this order
* `file_path`: the path to the CSV file where data should be saved. If the file doesn't exist, it will be created.

In [ ]:
import csv
import os
import json

def save_results_to_csv(result, features, file_path):
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        row = [result.get(feature, "") for feature in features]
        writer.writerow(row)


def converting_from_csv_to_json(features, csv_path, json_path):
    results = []
    with open(csv_path, 'r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)

        for row in csv_reader:

            result = {feature: value for feature, value in zip(features, row)}
            results.append(result)

    with open(json_path, 'w', encoding='utf-8') as json_file:
        json.dump(results, json_file, indent=4, ensure_ascii=False)


**Purpose**:
Loads a list of JSON objects from a JSON file.
**Parameters**:
* `file_path`: A string representing the path to the JSON file


In [ ]:
def reloading_result(file_path):
    with open(file_path, 'r') as file:
      results = json.load(file)

    return results

In [ ]:
datasets = loading_dataset(dataset_list, dataset_id)
models_and_tokenizers = loading_model(models_list, checkpoint_list)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

(…)d_posts_train_11552.split_with_val.jsonl:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

(…)gold_posts_dev_1444.split_with_val.jsonl:   0%|          | 0.00/2.70M [00:00<?, ?B/s]

(…)old_posts_test_1445.split_with_val.jsonl:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11552 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15885 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1445 [00:00<?, ? examples/s]

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-hf.
403 Client Error. (Request ID: Root=1-686577b9-4dd5397752e634c61553f22c;0fc6d49c-816f-4666-bab5-6cce5d9c052c)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-hf is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-2-7b-hf to ask for access.

In [ ]:
# amateur model
amateur = models_and_tokenizers[0]["model"]
amateur_tokenizer = models_and_tokenizers[0]["tokenizer"]

In [ ]:
# base model
base = models_and_tokenizers[1]["model"]
base_tokenizer = models_and_tokenizers[1]["tokenizer"]

In [ ]:
# unpack gsm8k dataset

gsm8k_train_dataloader, gsm8k_test_dataloader, gsm8k_debug_dataloader = datasets[0]

# unpack common gen dataset

common_gen_train_dataloader, common_gen_test_dataloader, common_gen_debug_dataloader = datasets[1]

# unpack what's that book dataset

wtb_train_dataloader, wtb_test_dataloader, wtb_debug_dataloader = datasets[2]


# Check if gsm8k is loaded properly
print(gsm8k_train_dataloader.features)
print_first_features_from_dataset(gsm8k_train_dataloader, 'question', 'answer')

# Check if common_gen is loaded properly
print(common_gen_train_dataloader.features)
# print_first_features_from_dataset(common_gen_train_dataloader, 'question', 'answer')

# Check if gsm8k is loaded properly
print(wtb_train_dataloader.features)
# print_first_features_from_dataset(wtb_train_dataloader, 'question', 'answer')

In [ ]:
import torch

def set_device():
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Using device: {device}")

  ## uncomment when having enough gpu
  # base_model.to(device)
  # amateur_model.to(device)
  # expert_model.to(device)
  return device

## GPT API feedback model designed for high-quality data generation.

In [ ]:
!pip install openai

In [ ]:
import openai
openai.api_key = "sk-xxxxxxxx"

The `GPT4oFeedbackEvaluator` class serves as a fine-tuning dataset generator for training both *expert* and *amateur* feedback models. It utilizes GPT-4o-mini to simulate the expert-level evaluation and critique of text responses. Operating in a two-phase pipelien, the class first generates a numeric **score** and **feedback** for a given (prompt, answer) pair. In the second phase, it produces an **improved answer** by conditioning on the original answer and generated feedback.

The outputs - original prompts, answers, feedback, scores, and improved responses - are stored in a structured format suitable for supervised fine-tuning. This makes the class essential for preparing high-quality datasets used in *knowledge distillation*, *self-critique*, and *student-teacher* architectures where the goal is to teacher smaller models to emulate expert behavior

In [ ]:
import json
import time
import re

class GPT4oFeedbackEvaluator:
    def __init__(self, model_name="gpt-4o-mini"):
        self.model_name = model_name
        self.expert_datasets = []


    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert evaluator. Your task is to score the quality of the following answer based "
                "on how well it responds to the given prompt.\n\n"
                "Use the following detailed scoring scale:\n"
                "1.0: Perfect completion with no issues.\n"
                "0.5 – 0.9: Generally good, minor flaws.\n"
                "0.1 – 0.4: Partially helpful, moderate issues.\n"
                "0.0: Neutral or off-topic, neither helpful nor harmful.\n"
                "-0.1 – -0.4: Mildly harmful or misleading.\n"
                "-0.5 – -0.9: Significantly off-task or misleading.\n"
                "-1.0: Complete failure or harmful content.\n\n"
                "Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.\n"
                "Do not comment on the length or verbosity of the answer.\n\n"
                "Respond exactly in this format:\n"
                "[reason]xxxx (MAX 50 words). Score: [score]\n\n"
                f"Prompt: {prompt}\n"
                f"Answer: {answer}\n"
            )
        }



        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}\n"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0,
                max_tokens=150,
                n=1
            )

            text = response.choices[0].message.content.strip()
            # print(f"Raw output:\n{text}")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", text, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract coverage and reasoning feedback (entire line before Score)
            feedback_line = text.split("Score")[0].strip()
            feedback = feedback_line

            # Store in expert_datasets
            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Failed to parse score or feedback. Error: {e}"

        return score, feedback


    def generate_improved_answer(self, prompt, original_answer, feedback):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert assistant. Your task is to read the prompt, the given answer, "
                "and the feedback, then generate an improved, accurate, and well-written answer that "
                "addresses the prompt fully and follows the feedback."
            )
        }
        user_message = {
            "role": "user",
            "content": (
                f"Prompt: {prompt}\n"
                f"Original Answer: {original_answer}\n"
                f"Feedback: {feedback}\n\n"
                "Please provide an improved answer."
            )
        }
        response = openai.chat.completions.create(
            model=self.model_name,
            messages=[system_message, user_message],
            temperature=0.7,
            max_tokens=300,
        )
        improved_answer = response.choices[0].message.content.strip()
        return improved_answer



In [ ]:
gpt_model = GPT4oFeedbackEvaluator()

In [ ]:
score, feedback = gpt_model.predict_score_and_feedback(
    prompt_text="Why do you like machine learning?",
    answer=(
        "Machine learning fascinates me"
    )
)

In [ ]:
print(feedback)
print(score)

[The response is vague and lacks depth, providing no specific reasons or insights into why machine learning is appealing.]
-0.19


## Setting up Expert Model

### 1k Sample Data Generation

**Dataset Setup & Description**
This script prepares a fine-tuning dataset for a feedback-based-self improvement model using the **Natural Questions** dataset from HuggingFace. The core idea is to simulate a learning process where a model not only evaluates answers but also improves them based on feedback - mimicking expert review behavior.

**Original Dataset**:

* Source: `sentence-transformers/natural-questions`
* Subset: 1500 randomly shuffled samples from the `train` split
* Fields Used:
  * `query`: Used as the prompt
  * `answer`: Used as the reference answer


**Data Generation Pipeline**
1. Corruption Phase (40% probability)
* The reference answer is intentionally corrupted using the `corrupt_answer()` function, which simulates weak or confused answers by:
  * Truncating sentences
  * Adding uncertainty
  * Altering numeric values

2. Evaluation Phase:
* A GPT-based model is queried with the prompt and (possibly corrupted) answer
* It generates:
  * A score (numeric rating)
  * Feedback (natural language critique)

3. Improvement Phase:
* Using the prompt, original (or corrupted) answer, and feedback, the model generates an improved answer

4. Storage:
* Each processed entry includes:
  * `prompt`
  * `original_answer`
  * `feedback`
  * `score`
  * `improved_answer`

**Dataset Split**
* Training Set:
  * `train_set = train_nq.select(range(1000))
  * First 1000 entries
  * Used for fine-tuning feedback-aware models

* Test Set:
  * `test_set = train_nq.select(range(1200, 1500))
  * Held-out 300 entries
  * Used for evaluating model generation

In [ ]:
import random
import re

def corrupt_answer(answer):
    answer = answer.strip()
    if random.random() < 0.3:
        return random.choice([
            "I'm not sure about that.",
            "It might be something else.",
            "Maybe it's related to that topic.",
            "I don't have a clear answer.",
            "It depends on the context."
        ])
    if random.random() < 0.3:
        return " ".join(answer.split()[:2]) + " ..."

    swapped = re.sub(r"\b(\d{4})\b", lambda m: str(int(m.group(1)) + random.choice([-10, 10, 100])), answer)
    if swapped != answer:
        return swapped
    return answer + ", I think."

In [ ]:
from datasets import load_dataset, DatasetDict

dataset = load_dataset("sentence-transformers/natural-questions")
train_nq = dataset["train"]

In [ ]:
train_nq = train_nq.shuffle(seed=42).select(range(1500))

train_set = train_nq.select(range(1000))
test_set = train_nq.select(range(1200, 1500))

In [ ]:
import pprint

processed_datasets = []

for item in train_nq:
    prompt = item["query"]
    answer = item["answer"]

    if random.random() < 0.4:
        original_answer = corrupt_answer(answer)
        score, feedback = gpt_model.predict_score_and_feedback(prompt, original_answer)
        improved_answer = gpt_model.generate_improved_answer(prompt, original_answer, feedback)
    else:
        original_answer = answer
        score, feedback = gpt_model.predict_score_and_feedback(prompt, original_answer)
        improved_answer = gpt_model.generate_improved_answer(prompt, original_answer, feedback)

    example = {
        "prompt": prompt,
        "original_answer": original_answer,
        "feedback": feedback,
        "score" : score,
        "improved_answer": improved_answer
    }

    processed_datasets.append(example)

    pprint.pprint(example)

In [ ]:
len(processed_datasets)

Save the data to json file

In [ ]:
import json

output_path = "processed_feedback_dataset.jsonl"

with open(output_path, "w") as f:
    for example in processed_datasets:
        json_line = json.dumps(example)
        f.write(json_line + "\n")

print(f"Saved dataset to {output_path}")

NameError: name 'processed_datasets' is not defined

Extract the data from the json file

In [ ]:
import json

file_path = "1k_dataset.jsonl"

dataset = []
with open(file_path, "r") as f:
    for line in f:
        example = json.loads(line)
        dataset.append(example)

print(dataset[0]["prompt"])
print(dataset[0]["original_answer"])
print(dataset[0]["feedback"])
print(dataset[0]["improved_answer"])

who are the basques and where do they live
Basques The Basques (/bɑːsks/ or /bæsks/; Basque: euskaldunak [eus̺kaldunak]; Spanish: vascos [ˈbaskos]; French: basques [bask]) are an indigenous ethnic group[5][6][7] characterised by the Basque language, a common culture and shared ancestry to the ancient Vascones and Aquitanians.[8] Basques are indigenous to and primarily inhabit an area traditionally known as the Basque Country (Basque: Euskal Herria), a region that is located around the western end of the Pyrenees on the coast of the Bay of Biscay and straddles parts of north-central Spain and south-western France.
The answer is accurate, complete, and well-written. It provides a detailed explanation of who the Basques are, including their language, culture, and ancestry. It also clearly states where they primarily live, which is in the Basque Country, a region around the western end of the Pyrenees on the coast of the Bay of Biscay, spanning parts of Spain and France.
The Basques are an

### Sample Dataset Entry

**Prompt:**  
`who are the basques and where do they live`

**Original Answer:**  
> The Basques (/bɑːsks/ or /bæsks/; Basque: euskaldunak; Spanish: vascos; French: basques) are an indigenous ethnic group characterized by the Basque language, a common culture, and shared ancestry to the ancient Vascones and Aquitanians.  
> They primarily inhabit the Basque Country (Euskal Herria), a region around the western end of the Pyrenees, straddling parts of north-central Spain and southwestern France.

**Score:** `1.0`

**Feedback:**  
> The answer is accurate, complete, and well-written. It provides a detailed explanation of who the Basques are...  

**Improved Answer:**  
> The Basques are an indigenous ethnic group primarily living in the Basque Country... They are characterized by their unique language, Basque, and a shared ancestry tracing back to the ancient Vascones and Aquitanians.


### Expert Model

In [ ]:
train_dataset = []
for example in dataset:
    prompt = example["prompt"]
    answer = example["original_answer"]
    feedback = example["feedback"]
    score = example["score"]

    train_dataset.append({
        "prompt" : prompt,
        "original_answer" : answer,
        "feedback" : feedback,
        "score" : score
    })

In [ ]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

The `ExpertClass` is a fine-tuning wrapper designed to train and evaluate a **feedback-generating** LLM that behaves like an expert evaluator. This class enables a large language model to:
1. Evaluate Answer Quality: Given a prompt and an answer, the model is trained to produce:
  * A numerical score (range from -1 to 1)
  * Concise feedback justifying that score

2. Fine-tune using ground-truth feedback:
* The model is fine-tuned using a combination of:
  * Language modeling loss (MSE) on the ground truth score and feedback
  * Semantic alignment (via BERT Score) between the model's generated feedback and human-annotated feedback

* A weighted sum of both loss types encourages both syntactic correctness and semantic alignment

3. Data Formatting & Tokenization:
* Prompts, answers, and feedbacks are wrapped in a standardized format for consistent training
* Token masking ensures loss is only computed after the `Score:` token, isolating the feedback generation

4. Training Utilities:
* Implement optimization (`AdamW`), gradient clipping, device configuration, and tokenizer handling
* Periodic logging of model outputs for debugging and qualitative inspection

5. Evaluation API:
* The `generate_feedback()` method enables inference-time use of the model to predict score + feecback for new samples

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from bert_score import score as bert_score
import re


class ExpertClass:
    def __init__(self, model_name="meta-llama/Meta-Llama-3-8B-Instruct", model_path=None, learning_rate=5e-5):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path if model_path else model_name,
            torch_dtype=self.model_dtype,
            device_map="auto"
        )

        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=learning_rate)
        self.loss_fn = nn.MSELoss()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def prepare_inputs_and_labels(self, prompt, answer, score, feedback):
        score = str(score)
        full_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
            f"Score: {score}\n"
            f"Feedback: {feedback}\n"
        )

        tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096)
        input_ids = tokenized.input_ids[0].clone()
        labels = input_ids.clone()

        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        score_token_ids = torch.tensor(score_token_ids, device=input_ids.device)

        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i + len(score_token_ids)], score_token_ids):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0

        labels[:start_idx] = -100
        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            tokenized.attention_mask.to(self.device)
        )

    def generate_feedback(self, prompt, answer, max_new_tokens=150):
        self.model.eval()

        input_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        pattern = r"Score:\s*([-+]?\d*\.?\d+)\s*Feedback:\s*(.*)"
        match = re.search(pattern, generated_text, re.DOTALL)

        if match:
            score = match.group(1).strip()
            feedback = match.group(2).strip().split("\n")[0]
            return f"Score: {score}\nFeedback: {feedback}"
        else:
            return generated_text.strip()

    def fine_tuning(self, train_dataset, output_dir, stopped=100, epochs=1, print_every=10, lm_loss_weight=0.2, semantic_loss_weight=0.8):
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0.0

            for i, sample in enumerate(train_dataset):
                if i == stopped:
                    break

                prompt = sample.get("prompt", "")
                answer = sample.get("original_answer", "")
                score = sample.get("score", "")
                feedback = sample.get("feedback", "")

                input_ids, labels, attention_mask = self.prepare_inputs_and_labels(prompt, answer, score, feedback)

                outputs = self.model(
                    input_ids=input_ids,
                    labels=labels,
                    attention_mask=attention_mask
                )
                lm_loss = outputs.loss

                predicted_feedback = self.generate_feedback(prompt, answer)
                P, R, F1 = bert_score([predicted_feedback], [feedback], lang="en", verbose=False)
                semantic_loss = 1.0 - F1[0].item()
                semantic_loss = torch.tensor(semantic_loss, dtype=self.model_dtype, device=self.device)

                total_combined_loss = lm_loss_weight * lm_loss + semantic_loss_weight * semantic_loss

                self.optimizer.zero_grad()
                total_combined_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += total_combined_loss.item()

                if (i + 1) % print_every == 0:
                    print(f"\n--- Sample {i+1} (Epoch {epoch+1}) ---")
                    print(f"Prompt: {prompt}")
                    print(f"Answer: {answer}")
                    print(f"GT Score: {score}")
                    print(f"GT Feedback:\n{feedback}")
                    print(f"\nGenerated Feedback:\n{predicted_feedback}")
                    print(f"LM Loss: {lm_loss.item():.4f}")
                    print(f"Semantic Loss: {semantic_loss.item():.4f}")
                    print(f"Total Loss: {total_combined_loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / (i + 1)
            print(f"\nEpoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}\n")

        # Save the fine-tuned model and tokenizer
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)

In [ ]:
expert_feedback_model = ExpertClass()

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
expert_output_dir = "expert_feedback_model"
expert_feedback_model.fine_tuning(train_dataset, expert_output_dir, stopped = 20)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 102.88 MiB is free. Process 38560 has 39.45 GiB memory in use. Of the allocated memory 38.78 GiB is allocated by PyTorch, and 164.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### [**Fine-tuned expert feedback model**](https://huggingface.co/henrytran07/expert_feedback_model/tree/main)

## Setting up Base Model

In [ ]:
prompt = "What are the benefits of being a machine learning engineer?"

In [ ]:
cot = """Q: California is nice.
A: California is renowned for its diverse geography and pleasant climate. From the sun-drenched beaches to the towering redwoods, the state offers incredible natural beauty. Its vibrant cities like Los Angeles and San Francisco attract millions with their culture and opportunities.

Q: {replace with the prompt}"""

### Processing the data before fine-tuning

In [ ]:
import json

file_path = "1ktrain_dataset.jsonl"
train_data = []

with open(file_path, "r") as f:
    for line in f:
        item = json.loads(line)

        prompt = item["prompt"]
        original_answer = item["original_answer"]
        improved_answer = item["improved_answer"]
        feedback = item["feedback"]
        train_data.append((prompt, original_answer, feedback, improved_answer))

print(f" {len(train_data)} examples loaded")

 1000 examples loaded


In [ ]:
from torch.utils.data import Dataset, DataLoader

class FeedbackDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

**Mock Feedback Models and Debugging Framework**

This set of classes and logic is designed to **simulate expert and amateur feedback generation** using OpenAI models for the purpose of debugging, evaluation, and iterative development of feedback-based training systems. These components are not part of the final production system but serve as scaffolding to support prototyping and testing.

1. `BaseMockFeedbackModel`
A base class providing a shared interface for all mock feedback models. It defines:
* A placeholder for feedback logic (`predict_score_and_feedback`, to be overriden)
* A basic revisiion method (`generate_improved_answer`) to mock response generation

2. `GPTExpertFeedbackModel`
Simulates an expert-level evaluator using `gpt-4o`or similar models. Key traits:
* Evaluates responses based on a nuanced scoring scale (-1.0 to 1.0)
* Returns structured reasoning and a score
* Logs expert-labeled examples for inspection or downstream use

3. `GPTAmateurFeedbackModel`
Simulates an amateur evaluator using `gpt-3.5-turbo` with a simpler, less technical scoring rubric:
* Emphasizes relatablity and general helpfulness
* Uses casual language to provide lightweight feedback

4. `MockAmateurExpertFeedbackNetwork`
Provides a simple coordination mechanism to combine feedback from both models:
* Generates a weighted score favoring the expert (adjustable with `alpha`)
* Merges both feedback sources into a single message
* Useful for testing how feedback fusion could work in training loops or agent collaboration


**Purpose**
These mock models:
* Stand in for real trained models during early testing
* Allow controlled debugging and development of data pipelines, scoring schemes, and fine-tuning procedures.
* Facilitate quick experimentation without needing a fully fine-tuned feedback model in place.

In [ ]:
import random
import re

class BaseMockFeedbackModel:
    def __init__(self, name="base"):
        self.name = name
        self.expert_datasets = []

    def predict_score_and_feedback(self, prompt_text, answer):
        # Override this in subclasses
        raise NotImplementedError

    def generate_improved_answer(self, prompt, original_answer, feedback):
        return f"[{self.name.upper()}] Improved answer based on feedback: {feedback}"


In [ ]:
import openai
import re

class GPTExpertFeedbackModel(BaseMockFeedbackModel):
    def __init__(self, model_name="gpt-4o"):
        super().__init__(name="expert")
        self.model_name = model_name

    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert evaluator. Your task is to score the quality of the following answer based "
                "on how well it responds to the given prompt.\n\n"
                "Use the following detailed scoring scale:\n"
                "1.0: Perfect completion with no issues.\n"
                "0.5 – 0.9: Generally good, minor flaws.\n"
                "0.1 – 0.4: Partially helpful, moderate issues.\n"
                "0.0: Neutral or off-topic, neither helpful nor harmful.\n"
                "-0.1 – -0.4: Mildly harmful or misleading.\n"
                "-0.5 – -0.9: Significantly off-task or misleading.\n"
                "-1.0: Complete failure or harmful content.\n\n"
                "Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.\n"
                "Respond exactly in this format:\n"
                "[reason]xxxx (MAX 50 words). Score: [score]\n\n"
            )
        }

        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0,
                max_tokens=150,
                n=1
            )
            output = response.choices[0].message.content.strip()
            print(f"[Expert Model Feedback]\n{output}\n")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", output, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract feedback (text before "Score")
            feedback = output.split("Score")[0].strip()

            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Error parsing GPT expert feedback: {e}"

        return score, feedback


In [ ]:
import openai
import re

class GPTAmateurFeedbackModel(BaseMockFeedbackModel):
    def __init__(self, model_name="gpt-3.5-turbo"):
        super().__init__(name="amateur")
        self.model_name = model_name

    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an amateur evaluator learning how to assess answers. Your job is to give simple feedback "
                "and assign a rough score based on how helpful or understandable the answer is.\n\n"
                "Use this scoring scale:\n"
                "1.0: Great answer, easy to understand.\n"
                "0.5 – 0.9: Pretty good, some things could be clearer.\n"
                "0.1 – 0.4: Okay, but a bit confusing or missing things.\n"
                "0.0 or below: Not helpful, or doesn't make sense.\n\n"
                "Try your best to explain in everyday language what worked and what didn't.\n"
                "Respond in this format:\n"
                "[short feedback] Score: [score]\n\n"
            )
        }

        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0.7,
                max_tokens=150
            )

            output = response.choices[0].message.content.strip()
            print(f"[Amateur Model Feedback]\n{output}\n")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", output, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract feedback
            feedback = output.split("Score")[0].strip()

            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Error parsing GPT amateur feedback: {str(e)}"

        return score, feedback

In [ ]:
class MockAmateurExpertFeedbackNetWork:
    def __init__(self):
        self.teacher = GPTExpertFeedbackModel()
        self.student = GPTAmateurFeedbackModel()

    def generate_combined_feedback(self, prompt_text, answer, alpha=0.8):
        # Get individual scores and feedback
        student_score, student_feedback = self.student.predict_score_and_feedback(prompt_text, answer)
        teacher_score, teacher_feedback = self.teacher.predict_score_and_feedback(prompt_text, answer)

        # Weighted average score favoring expert
        combined_score = alpha * teacher_score + (1 - alpha) * student_score

        # Combined text (for revision prompt)
        combined_feedback = (
            f"The expert highlights: {teacher_feedback}\n"
            f"The amateur also adds: {student_feedback}"
        )

        return {
            "combined_score": combined_score,
            "combined_feedback": combined_feedback,
            "expert_feedback": teacher_feedback,
            "amateur_feedback": student_feedback
        }

This `BaseModelGPT` class is designed to generate and refine high-quality training data for a supervised fine-tuning pipeline involving two types of feedback models: an expert and an amateur. It simulates a self-improvement framework for language models, where an initial answer to a given prompt is evaluated by these two feedback sources, and then revised iteratively until it reaches a desirable level of quality. The goal is to emulate a learning environment where feedback is continuously incorporated into improved model outptus, ultimately producing fine-tuning data that reflect both expert knowledge and layperson intuition.

At its core, the class perform several critical functions:
* It receives a question and an initial answer, either generated by a base model or supplied by the user
* It sends this answer to a feedback network, which combines evaluation scores and written critiques from both an expert and an amateur model
* Based on these evaluations, the class constructs a refined prompt that instructs the model to revise its answer in a way that prioritizes the most meaningful feedback especially the expert's
* After generating a new, revised answer, the class then checks whether the revision sufficiently addresses the key concerns raised by the feedback.
* It calculates a reward score that reflects how much the new answer has improved compared to the previous one. If the answer fails to improve or regresses in quality, a penalty is applied
* The loop repeats: the model continues revising its answer, integrating fresh feedback each time, until either a target quality threshold is reached (as defined by the score) or a maximum number of refinement iterations is met.



In [ ]:
import openai
import re

openai.api_key = "sk-xxxxxxxxx"

class BaseModelGPT:
    def __init__(self, model_name, network, penalty_factor=1.0):
        self.model_name = model_name
        self.network = network
        self.penalty_factor = penalty_factor
        self.last_score = 0.0

    def generate_prompt(self, question, previous_answer, expert_feedback, amateur_feedback, cot=None):
        prompt = ""
        if cot:
            prompt += cot.replace("{replace with the prompt}", question.strip()) + "\n\n"
        else:
            prompt += f"Q: {question.strip()}\n"

        if previous_answer:
            if isinstance(previous_answer, tuple):
                previous_answer = previous_answer[1]
            prompt += f"Previous Answer: {previous_answer.strip()}\n"

        prompt += (
            f"\nThe previous answer received a score of {self.last_score:.2f} on a scale from -1.0 to 1.0, "
            "where:\n"
            "-1.0 = completely incorrect or misleading\n"
            " 0.0 = neutral or partially helpful\n"
            " 1.0 = accurate, complete, and helpful\n\n"
            "Two evaluators have provided feedback on the answer:\n"
            f"- **Expert Feedback** (weighted more heavily): {expert_feedback.strip()}\n"
            f"- **Amateur Feedback**: {amateur_feedback.strip()}\n\n"
            "Your task is to revise the previous answer, **giving more weight to the expert feedback** while still addressing any valuable points raised in the amateur feedback.\n\n"
            "Please ensure the revised answer:\n"
            "- Fully addresses all issues raised (especially those in the expert feedback)\n"
            "- Improves clarity, accuracy, and structure\n"
            "- Preserves useful parts of the original answer\n"
            "- Is logically complete and ends with a strong conclusion\n\n"
            "**Strive to improve the overall quality and score of the answer.**\n\n"
            "Revised Answer:\nA: "
        )

        return prompt

    def generate_answer(self, question, previous_answer=None, previous_feedback=None, cot=None, previous_score=0.0, max_tokens=200):
        self.last_score = previous_score

        result = self.network.generate_combined_feedback(question, previous_answer)
        combined_score = result["combined_score"]
        combined_feedback = result["combined_feedback"]
        expert_feedback = result["expert_feedback"]
        amateur_feedback = result["amateur_feedback"]

        full_prompt = self.generate_prompt(
            question,
            previous_answer,
            expert_feedback=expert_feedback,
            amateur_feedback=amateur_feedback,
            cot=cot
        )

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that improves answers based on feedback."},
                    {"role": "user", "content": full_prompt}
                ],
                temperature=0.7,
                max_tokens=max_tokens
            )
            answer = response.choices[0].message.content.strip()
            print(f"\nGenerated Answer:\n{answer}\n")

            return combined_feedback, answer, combined_score

        except Exception as e:
            print(f"Error generating answer: {e}")
            return "API Error", "API Error", 0.0

    def check_feedback_satisfaction(self, question, previous_answer, feedback, current_answer):
        review_prompt = (
            f"Question: {question}\n\n"
            f"Previous Answer: {previous_answer}\n\n"
            f"Combined Feedback (from expert and amateur evaluators): {feedback}\n\n"
            f"Current Answer: {current_answer}\n\n"
            "Based on the combined feedback, does the current answer appropriately address the most important points raised?\n"
            "Consider both expert and amateur feedback but prioritize the expert perspective. "
            "Your reply should begin with 'Yes' or 'No'. If 'No', briefly explain what is still missing or unclear."
        )

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a critical reviewer assessing if feedback was properly incorporated."},
                    {"role": "user", "content": review_prompt}
                ],
                temperature=0.5,
                max_tokens=150,
            )

            output = response.choices[0].message.content.strip()
            print(f"Feedback Satisfaction Check:\n{output}\n")
            return output.lower().startswith("yes"), output

        except Exception as e:
            print(f"OpenAI API error: {e}")
            return False, "API Error"


    def update_model_with_feedback(self, question, previous_answer, previous_feedback, cot=None):
        combined_feedback, improved_answer, improved_score = self.generate_answer(
            question, previous_answer, previous_feedback, cot
        )

        prev_score = self.network.generate_combined_feedback(question, previous_answer)["combined_score"]

        delta = improved_score - prev_score
        reward = delta if delta >= 0 else delta * self.penalty_factor

        min_reward_magnitude = 0.05
        if abs(reward) < min_reward_magnitude and reward != 0:
            reward = reward / abs(reward) * min_reward_magnitude

        is_satisfied, critique = self.check_feedback_satisfaction(
            question, previous_answer, combined_feedback, improved_answer
        )
        reward += 0.2 if is_satisfied else -0.2

        print(f"Previous Score: {prev_score:.4f}, Improved Score: {improved_score:.4f}, Reward: {reward:.4f}")
        print(f"Critique: {critique}\n")

        return improved_answer, combined_feedback, improved_score, reward

    def self_critique(self, prompt, previous_answer, previous_feedback, cot=None, iterations=10, target_score=0.95):
        best_answer = previous_answer
        feedback = previous_feedback
        best_score = self.network.generate_combined_feedback(prompt, previous_answer)["combined_score"]

        previous_answers = set()
        previous_answers.add(best_answer)

        for step in range(iterations):
            if best_score >= target_score:
                print(f"Target score reached ({best_score:.4f}), stopping early.")
                break

            improved_answer, feedback, improved_score, reward = self.update_model_with_feedback(
                prompt, best_answer, feedback, cot
            )

            if improved_answer in previous_answers:
                reward -= 0.1
                print("Penalty: Repeated answer detected.")

            previous_answers.add(improved_answer)

            print(f"Step {step}: Reward {reward:.4f}, Score {improved_score:.4f}")
            print(f"Improved Answer:\n{improved_answer}\n")

            if improved_score > best_score:
                best_answer = improved_answer
                best_score = improved_score

        print(f"Final Answer:\n{best_answer}")
        print(f"Final Score: {best_score:.4f}")
        return best_answer, best_score

In [ ]:
model_name = "gpt-3.5-turbo"
network = MockAmateurExpertFeedbackNetWork()
base_model = BaseModelGPT(model_name, network)

In [ ]:
penalty_factor = 2
base = BaseModel_Test(base_model, base_tokenizer, gpt_model, penalty_factor)

In [ ]:
base.fine_tuning()

NameError: name 'base' is not defined

In [ ]:
feedback_few_shot, answer_few_shot, score_few_shot = base_model.generate_answer(prompt)

[Amateur Model Feedback]
This answer is not helpful as it simply states "None" without providing any explanation or details. It would be more useful to mention some reasons why there may be benefits to being a machine learning engineer. 
Score: 0.0

[Expert Model Feedback]
[reason]The answer is uninformative and dismissive, failing to address the prompt's request for benefits. It does not provide any insights or information about the advantages of being a machine learning engineer, such as career opportunities, innovation, or problem-solving skills. Score: 0.0


Generated Answer:
Being a machine learning engineer offers numerous benefits that make it an exciting and rewarding career choice. Some of the advantages include:

1. **Career Opportunities**: Machine learning is a rapidly growing field with high demand for skilled professionals. As a machine learning engineer, you have access to a wide range of job opportunities in various industries, offering career stability and growth poten

In [ ]:
print(score_few_shot)

0.0


In [ ]:
print(feedback_few_shot)

The expert highlights: [reason]The answer is uninformative and dismissive, failing to address the prompt's request for benefits. It does not provide any insights or information about the advantages of being a machine learning engineer, such as career opportunities, innovation, or problem-solving skills.
The amateur also adds: This answer is not helpful as it simply states "None" without providing any explanation or details. It would be more useful to mention some reasons why there may be benefits to being a machine learning engineer.


In [ ]:
base_model.self_critique(prompt, answer_few_shot, feedback_few_shot, iterations = 10)

[Amateur Model Feedback]
This answer provides a clear and concise explanation of the benefits of being a machine learning engineer. It covers various advantages such as career opportunities, innovation, problem-solving skills, and competitive salaries. The information is well-structured and easy to understand.

Score: 1.0

[Expert Model Feedback]
The answer effectively outlines the benefits of being a machine learning engineer, covering career opportunities, innovation, skill development, and competitive salaries. However, it ends abruptly, suggesting there might be more to say about financial rewards. Overall, it is informative and relevant. Score: 0.9

[Amateur Model Feedback]
This answer provides clear and concise points on the benefits of being a machine learning engineer. It covers various aspects such as career opportunities, innovation, problem-solving skills, and competitive salaries. The language is easy to understand and the benefits are well-explained. However, it could be i

('Being a machine learning engineer offers numerous benefits that make it an exciting and rewarding career choice. Some of the advantages include:\n\n1. **Career Opportunities**: Machine learning is a rapidly growing field with high demand for skilled professionals. As a machine learning engineer, you have access to a wide range of job opportunities in various industries, offering career stability and growth potential.\n\n2. **Innovation**: Machine learning engineers are at the forefront of technological innovation. By developing and implementing cutting-edge algorithms and models, you can contribute to groundbreaking advancements in areas such as artificial intelligence, healthcare, finance, and more.\n\n3. **Problem-Solving Skills**: Working in machine learning requires strong analytical and problem-solving abilities. As you tackle complex data challenges and develop solutions using machine learning techniques, you enhance your critical thinking and decision-making skills.\n\n4. **Co

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_path = "/content/drive/MyDrive/finetuned_base_model"

base.model.save_pretrained(save_path)
base.tokenizer.save_pretrained(save_path)

NameError: name 'base' is not defined

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

save_path = "/content/drive/MyDrive/finetuned_base_model"

base_tokenizer = AutoTokenizer.from_pretrained(save_path)
base_model = AutoModelForCausalLM.from_pretrained(save_path)

## Amateur Feedback Model

The `AmateurFeedbackModel` class defines a lightweight language model training pipeline intended to simulate a non-expert feedback generator. Its primary function is to generate feedback and scalar evaluation scores on answers given a prompt. This model represents the "amateur" or "crowdsourced" layer of feedback in a broader system that also includes expert feedback, and is particularly useful in bootstrapping low-cost evaluation signals in multi-annotator training regimes.


This model is built using a compact, instruction-tuned language model (e.g. TinyLLaMA) and supports:
* Feedback Generation: Given a question and an answer, the model returns a scalar score (from -1 to 1) and a short feedback sentence judging the answer's quality
* Supervised Fine-Tuning (SFT): The model is fine-tuned on tuples of `(prompt, answer, score, feedback)` to learn to emulate scoring and feedback behavior.
* Hybrid Loss Optimization: Training incorporates a combination of:
  * Language Modeling Loss (LM Loss): Predicting the expected score and feedback tokens
  * Semantic Similarity Loss: Using BERTScore to compare the model's generated feedback with the ground-truth feedback, encouraging semantic alignment beyond surface token accuracy.


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from bert_score import score as bert_score
import re


class AmateurFeedbackModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", model_path=None, lr=5e-5):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path if model_path else model_name,
            torch_dtype=self.model_dtype,
            device_map="auto"
        )

        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def prepare_inputs_and_labels(self, prompt, answer, score, feedback):
        score = str(score)
        full_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
            f"Score: {score}\n"
            f"Feedback: {feedback}\n"
        )

        tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096)
        input_ids = tokenized.input_ids[0].clone()
        labels = input_ids.clone()

        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        score_token_ids = torch.tensor(score_token_ids, device=input_ids.device)

        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i + len(score_token_ids)], score_token_ids):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0

        labels[:start_idx] = -100

        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            tokenized.attention_mask.to(self.device)
        )

    def generate_feedback(self, prompt, answer, max_new_tokens=150):
        self.model.eval()

        input_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        pattern = r"Score:\s*([-+]?\d*\.?\d+)\s*Feedback:\s*(.*)"
        match = re.search(pattern, generated_text, re.DOTALL)

        if match:
            score = match.group(1).strip()
            feedback = match.group(2).strip().split("\n")[0]
            return f"Score: {score}\nFeedback: {feedback}"
        else:
            return generated_text.strip()

    def fine_tuning(self, train_dataset, output_dir, stopped=100, epochs=1, print_every=10, lm_loss_weight=0.2, semantic_loss_weight=0.8):
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0.0

            for i, sample in enumerate(train_dataset):
                if i == stopped:
                    break

                prompt = sample.get("prompt", "")
                answer = sample.get("original_answer", "")
                score = sample.get("score", "")
                feedback = sample.get("feedback", "")

                input_ids, labels, attention_mask = self.prepare_inputs_and_labels(prompt, answer, score, feedback)

                outputs = self.model(
                    input_ids=input_ids,
                    labels=labels,
                    attention_mask=attention_mask
                )
                lm_loss = outputs.loss

                predicted_feedback = self.generate_feedback(prompt, answer)
                P, R, F1 = bert_score([predicted_feedback], [feedback], lang="en", verbose=False)
                semantic_loss = 1.0 - F1[0].item()
                semantic_loss = torch.tensor(semantic_loss, dtype=self.model_dtype, device=self.device)

                total_combined_loss = lm_loss_weight * lm_loss + semantic_loss_weight * semantic_loss

                self.optimizer.zero_grad()
                total_combined_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += total_combined_loss.item()

                if (i + 1) % print_every == 0:
                    print(f"\n--- Sample {i+1} (Epoch {epoch+1}) ---")
                    print(f"Prompt: {prompt}")
                    print(f"Answer: {answer}")
                    print(f"GT Score: {score}")
                    print(f"GT Feedback:\n{feedback}")
                    print(f"\nGenerated Feedback:\n{predicted_feedback}")
                    print(f"LM Loss: {lm_loss.item():.4f}")
                    print(f"Semantic Loss: {semantic_loss.item():.4f}")
                    print(f"Total Loss: {total_combined_loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / (i + 1)
            print(f"\nEpoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}\n")

        # Save the fine-tuned model and tokenizer
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)

In [ ]:
amateur_feedback_model = AmateurFeedbackModel()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
amateur_feedback_model.fine_tuning(train_dataset)

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Sample 100 (Epoch 1) ---
Prompt: how many episode are there in season 4 of the originals
Answer: I can't really tell from this.
Ground Truth Score: -1.0
Ground Truth Feedback:
The response is completely irrelevant and does not provide any information about the number of episodes in season 4 of The Originals. Please provide a specific answer to the question.

Generated Feedback:
Score: -1.0
Feedback: The answer is completely incorrect and irrelevant. It does not provide any information about the number of episodes in the original series. Please provide a specific answer.
Loss: 0.6361
------------------------------------------------------------


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Sample 200 (Epoch 1) ---
Prompt: who voiced spider man in the new game
Answer: Spider-Man (2018 video game) Yuri Lowenthal is the voice actor and John Bubniak is the physical actor for Spider-Man in the game.[18] In early 2015, Lowenthal got the role after some initial pushback within Insomniac due to him voicing one of the main playable characters for Insomniac's most recently released game at the time, Sunset Overdrive, but ultimately the game's lead writer Jon Paquette convinced the studio to cast him in the role due to his trust in Lowenthal's acting ability.[18] Lowenthal worked with two stunt people throughout the game's development.[18] He tried to differentiate between his voices for Peter Parker and Spider-Man but thought that they could not be totally different and as a result spent a large amount of time "finessing and massaging" his performance to achieve a balance.[18] Jon Paquette, Christos Gage, Ben Arfmann, and Dan Slott serve as writers of the game.[1]
Ground Trut

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Sample 300 (Epoch 1) ---
Prompt: who wrote you're going to love me
Answer: And I ...
Ground Truth Score: -1.0
Ground Truth Feedback:
The answer is incomplete and does not provide any relevant information in response to the prompt. Please provide the name of the author or songwriter for the song "You're Going to Love Me".

Generated Feedback:
Score: -1.0
Feedback: The response is incomplete and does not provide any information about who wrote you're going to love me. Please provide a complete and accurate response.
Loss: 0.8796
------------------------------------------------------------


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Sample 400 (Epoch 1) ---
Prompt: is the movie se7en based on a book
Answer: I think it might be something else.
Ground Truth Score: -1.0
Ground Truth Feedback:
The answer is incorrect and vague. It does not provide a clear response to the question about whether the movie "Se7en" is based on a book.

Generated Feedback:
Score: -1.0
Feedback: The response is completely irrelevant and does not provide any information about the movie "Se7en". Please provide a specific and accurate answer to the question.
Loss: 0.8543
------------------------------------------------------------


KeyboardInterrupt: 

## Knowledge Distillation Network

In [ ]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [ ]:
import json

file_path = "1k_dataset.jsonl"
expert_datasets = []

with open(file_path, "r") as f:
    for line in f:
        item = json.loads(line)

        prompt = item["prompt"]
        original_answer = item["original_answer"]
        score = item["score"]
        feedback = item["feedback"]
        expert_datasets.append(
            {"prompt": prompt,
             "answer": original_answer,
             "feedback": feedback,
             "score": score}
        )

print(f" {len(expert_datasets)} examples loaded")

 1000 examples loaded


The `AmateurExpertFeedbackNetwork` class is a multi-model feedback training architecture that facilitates knowledge distillation from an expert model (teacher) to a student model (amateur). It is desgined to refine the student model's ability to generate human-like feedback on open-domain question answering tasks by learning from both expert annotations and the teacher model's behavior

This framework orchestrates the knowledge transfer process from a stronger, more informed model (teacher) to a smaller or initially weaker (student) via:
* Supervised fine-tuning (SFT) on expert-annotated datasets
* Feedback imitation through semantic similarity scoring
* Cosine similarity based hidden state alignment
* A carefully weighted loss function that blends multiple objectives (language modeling, score regression, hidden alignment, and semantic consistency)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from bert_score import score as bert_score
import re


class AmateurExpertFeedbackNetWork:
    def __init__(self, student, teacher, expert_datasets=None, device=None, optimizer=None):
        self.student = student
        self.teacher = teacher
        self.expert_datasets = expert_datasets or []
        self.student_tokenizer = student.tokenizer
        self.teacher_tokenizer = teacher.tokenizer
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        if self.student_tokenizer.pad_token is None:
            self.student_tokenizer.pad_token = self.student_tokenizer.eos_token

        self.optimizer = optimizer if optimizer else AdamW(self.student.model.parameters(), lr=5e-5)
        self.align_hidden = None

    def knowledge_distillation_using_SFT(self, epochs=3, stopped=None, warmup_alpha=False):
        if len(self.expert_datasets) % 100 == 0 and len(self.expert_datasets) > 0:
            print(f"Starting fine-tune on {len(self.expert_datasets)} samples")

        score_loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            print(f"\n--- Epoch {epoch + 1}/{epochs} ---")
            total_loss_accum = 0

            for i, expert_sample in enumerate(self.expert_datasets):
                if stopped is not None and i == stopped:
                    break

                prompt_text = expert_sample.get("prompt")
                answer = expert_sample.get("answer")
                expert_score = expert_sample.get("score")
                expert_feedback = expert_sample.get("feedback")

                if not prompt_text or not answer or expert_score is None:
                    continue

                print(f"\nSample {i + 1}")

                input_ids, labels, attention_mask = self.student.prepare_inputs_and_labels(
                    prompt_text, answer, expert_score, expert_feedback
                )

                student_out = self.student.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    output_hidden_states=True,
                    return_dict=True
                )
                lm_loss = student_out.loss
                student_hidden = student_out.hidden_states[-1][:, 0, :]

                with torch.no_grad():
                    student_output = self.student.generate_feedback(prompt_text, answer)
                    student_score_match = re.search(r"Score:\s*([-+]?\d*\.?\d+)", student_output)
                    student_feedback_match = re.search(r"Feedback:\s*(.*)", student_output)

                    try:
                        student_score = float(student_score_match.group(1)) if student_score_match else 0.0
                    except ValueError:
                        student_score = 0.0

                    student_feedback = student_feedback_match.group(1).strip() if student_feedback_match else student_output.strip()

                    print(f"Student Feedback Only:\n  Score   : {student_score}\n  Feedback: {student_feedback}")

                with torch.no_grad():
                    teacher_output_full = self.teacher.generate_feedback(prompt_text, answer)
                    teacher_score_match = re.search(r"Score:\s*([-+]?\d*\.?\d+)", teacher_output_full)
                    teacher_feedback_match = re.search(r"Feedback:\s*(.*)", teacher_output_full)

                    try:
                        teacher_score = float(teacher_score_match.group(1)) if teacher_score_match else 0.0
                    except ValueError:
                        teacher_score = 0.0

                    teacher_feedback = teacher_feedback_match.group(1).strip() if teacher_feedback_match else teacher_output_full.strip()

                    print(f"Teacher Feedback Only:\n  Score   : {teacher_score}\n  Feedback: {teacher_feedback}")

                    teacher_inputs = self.teacher_tokenizer(
                        prompt_text, return_tensors="pt", truncation=True, padding=True
                    ).to(self.device)

                    teacher_out = self.teacher.model(
                        input_ids=teacher_inputs["input_ids"],
                        attention_mask=teacher_inputs["attention_mask"],
                        output_hidden_states=True,
                        return_dict=True
                    )
                    teacher_hidden = teacher_out.hidden_states[-1][:, 0, :]

                student_hidden = student_hidden.to(self.model_dtype)
                teacher_hidden = teacher_hidden.to(self.model_dtype)

                if self.align_hidden is None:
                    student_dim = student_hidden.size(-1)
                    teacher_dim = teacher_hidden.size(-1)
                    self.align_hidden = (
                        nn.Linear(student_dim, teacher_dim).to(self.device).to(self.model_dtype)
                        if student_dim != teacher_dim else nn.Identity()
                    )

                student_aligned = self.align_hidden(student_hidden)
                cos_sim = nn.functional.cosine_similarity(student_aligned, teacher_hidden, dim=-1)
                hidden_loss = 1.0 - cos_sim.mean()

                student_score_tensor = torch.tensor([student_score], dtype=torch.float32, device=self.device)
                expert_score_tensor = torch.tensor([expert_score], dtype=torch.float32, device=self.device)
                score_loss = score_loss_fn(student_score_tensor, expert_score_tensor)

                P1, R1, F1 = bert_score([student_feedback], [expert_feedback], lang="en", verbose=False)
                P2, R2, F2 = bert_score([student_feedback], [teacher_feedback], lang="en", verbose=False)

                F1 = F1.to(self.device).to(self.model_dtype)
                F2 = F2.to(self.device).to(self.model_dtype)

                semantic_loss_expert = (1.0 - F1).mean()
                semantic_loss_teacher = (1.0 - F2).mean()


                teacher_f1 = bert_score([teacher_feedback], [expert_feedback], lang="en", verbose=False)[2][0].item()
                if teacher_f1 < 0.85:
                    print(f"Fine-tuning teacher (F1={teacher_f1:.3f}) due to low semantic match with expert")
                    input_text = (
                        "You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.\n"
                        "Evaluate the answer on a scale from -1 to 1, where:\n"
                        "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
                        "0 = neutral or partially addresses the prompt\n"
                        "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
                        f"Prompt: {prompt_text}\n"
                        f"Answer: {answer}\n"
                        f"Score: {expert_score}\n"
                        f"Feedback: {expert_feedback}\n"
                    )
                    tokens = self.teacher_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(self.device)
                    labels = tokens.input_ids.clone()
                    teacher_optimizer = AdamW(self.teacher.model.parameters(), lr=5e-6)
                    self.teacher.model.train()
                    teacher_outputs = self.teacher.model(
                        input_ids=tokens.input_ids,
                        attention_mask=tokens.attention_mask,
                        labels=labels
                    )
                    teacher_loss = teacher_outputs.loss
                    teacher_optimizer.zero_grad()
                    teacher_loss.backward()
                    teacher_optimizer.step()
                    print(f"Teacher fine-tuned on this sample (loss={teacher_loss.item():.4f})")

                alpha = (epoch / epochs * 0.2) if warmup_alpha else 0.2
                beta, gamma, delta, epsilon = 0.5, 1.0, 1.0, 0.5

                total_loss = (
                    alpha * lm_loss +
                    beta * score_loss +
                    gamma * semantic_loss_expert +
                    delta * hidden_loss +
                    epsilon * semantic_loss_teacher
                )

                self.optimizer.zero_grad()
                total_loss.backward()
                self.optimizer.step()

                print(f"LM Loss          : {lm_loss.item():.4f}")
                print(f"Score Loss       : {score_loss.item():.4f}")
                print(f"Semantic Loss (Student vs Expert) : {semantic_loss_expert.item():.4f}")
                print(f"Semantic Loss (Student vs Teacher): {semantic_loss_teacher.item():.4f}")
                print(f"Hidden Align Loss                : {hidden_loss.item():.4f}")
                print(f"Total Loss                       : {total_loss.item():.4f}")

                total_loss_accum += total_loss.item()

            avg_loss = total_loss_accum / len(self.expert_datasets) if self.expert_datasets else 0.0
            print(f"\nEpoch {epoch + 1} average loss: {avg_loss:.4f}")

        return self.teacher, self.student

    def generate_combined_feedback(self, prompt_text, answer, alpha=0.65, distill_threshold=0.90):
        # Student output parsing
        student_output = self.student.generate_feedback(prompt_text, answer)
        student_score_match = re.search(r"Score:\s*([-+]?\d*\.?\d+)", student_output)
        student_feedback_match = re.search(r"Feedback:\s*(.*)", student_output)

        try:
            student_score = float(student_score_match.group(1)) if student_score_match else 0.0
        except ValueError:
            student_score = 0.0

        student_feedback = student_feedback_match.group(1).strip() if student_feedback_match else student_output.strip()

        # Teacher output parsing
        teacher_output = self.teacher.generate_feedback(prompt_text, answer)
        teacher_score_match = re.search(r"Score:\s*([-+]?\d*\.?\d+)", teacher_output)
        teacher_feedback_match = re.search(r"Feedback:\s*(.*)", teacher_output)

        try:
            teacher_score = float(teacher_score_match.group(1)) if teacher_score_match else 0.0
        except ValueError:
            teacher_score = 0.0

        teacher_feedback = teacher_feedback_match.group(1).strip() if teacher_feedback_match else teacher_output.strip()

        # Semantic similarity check (Student vs Teacher)
        _, _, f1 = bert_score([student_feedback], [teacher_feedback], lang="en", verbose=False)
        f1_score = f1[0].item()

        if f1_score < distill_threshold:
            print(f"[Distill Trigger] Semantic similarity too low (F1={f1_score:.3f}) — initiating knowledge distillation.")
            max_distill_attempts = 5

            for attempt in range(max_distill_attempts):
                print(f"[Distill Attempt {attempt+1}]")
                self.knowledge_distillation_using_SFT(epochs=1, stopped=32)

                # New feedback after distillation
                new_student_output = self.student.generate_feedback(prompt_text, answer)
                new_student_feedback = re.search(r"Feedback:\s*(.*)", new_student_output)
                student_feedback_text = new_student_feedback.group(1).strip() if new_student_feedback else new_student_output.strip()

                new_teacher_output = self.teacher.generate_feedback(prompt_text, answer)
                new_teacher_feedback = re.search(r"Feedback:\s*(.*)", new_teacher_output)
                teacher_feedback_text = new_teacher_feedback.group(1).strip() if new_teacher_feedback else new_teacher_output.strip()

                _, _, f1_student = bert_score([student_feedback_text], [teacher_feedback_text], lang="en", verbose=False)
                f1_student_val = f1_student[0].item()

                print(f"[Check] F1 (Student vs Teacher): {f1_student_val:.3f}")

                if f1_student_val > distill_threshold:
                    print("[Success] Student improved sufficiently. Early stopping distillation.")
                    student_feedback = student_feedback_text
                    teacher_feedback = teacher_feedback_text
                    break
            else:
                print("[Warning] Max distillation attempts reached; student may not have improved sufficiently.")

        combined_score = alpha * teacher_score + (1 - alpha) * student_score
        combined_feedback = (
            f"The expert highlights: {teacher_feedback}\n"
            f"The student also adds: {student_feedback}"
        )

        print(f"Combined feedback: {combined_feedback}")
        return {
            "combined_score": combined_score,
            "combined_feedback": combined_feedback,
            "expert_feedback": teacher_feedback,
            "amateur_feedback": student_feedback
        }

In [ ]:
network = AmateurExpertFeedbackNetWork(base_model, expert_feedback_model, expert_datasets)
base_model.network = network

In [ ]:
test_prompt = "Why do you want to become a machine learning engineer?"
feedback, answer, score = base_model.generate_answer(test_prompt)
print(answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Starting fine-tune on 1000 samples

--- Epoch 1/3 ---

Sample 1


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Student Feedback Only:
  Score   : 1.0
  Feedback: The answer is accurate, complete, and well-written. It provides a detailed explanation of the Basque language, Basque, and Basque ancestry, Basque Country, and Basque Country, and also includes additional relevant information about the Basque language and Basque ancestry.
Teacher Feedback Only:
  Score   : 1.0
  Feedback: The answer is accurate, complete, and well-written. It provides a detailed explanation of the Basque language, Basque, and Basque ancestry, Basque Country, and Basque Country, and also includes additional relevant information about the Basque language and Basque ancestry.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LM Loss          : 0.3205
Score Loss       : 0.0400
Semantic Loss    : 0.1836
Hidden Align Loss: 1.0033
Total Loss       : 0.9340

Sample 8
Student Feedback Only:
  Score   : 0.0
  Feedback: You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.
Evaluate the answer on a scale from -1 to 1, where:
-1 = completely incorrect, irrelevant, or does not address the prompt
0 = neutral or partially addresses the prompt
1 = excellent, fully correct, detailed, and thoroughly addresses the prompt

Respond exactly in this format:
Prompt: who is the chief executive of the state of florida
Answer: Government of Florida The Governor of Florida is the chief executive of the government of Florida and the chief administrative officer of the state responsible for the planning and budgeting for the state, and serves as chair when the Governor and the Florida Cabinet sit as a decision-making body in various constitutional rol

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LM Loss          : 0.1806
Score Loss       : 0.2500
Semantic Loss    : 0.1598
Hidden Align Loss: 1.0032
Total Loss       : 0.8871

Sample 9
Student Feedback Only:
  Score   : 0.0
  Feedback: You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.
Evaluate the answer on a scale from -1 to 1, where:
-1 = completely incorrect, irrelevant, or does not address the prompt
0 = neutral or partially addresses the prompt
1 = excellent, fully correct, detailed, and thoroughly addresses the prompt

Respond exactly in this format:
Prompt: who play mary poppins in mary poppins returns
Answer: Mary Poppins ...


KeyboardInterrupt: 

**Build Dataset**

In [ ]:
def build_kd_dataset(results):
    kd_data = []
    for ex in results:
        inp = f"Question: {ex['input']}\nStudent Answer: {ex['x0']}"
        out = ex['p_expert']
        kd_data.append({"input": inp, "output": out})
    return kd_data

In [ ]:
#Saving FILE

import json
with open("kd_data.json", "w") as f:
    json.dump(build_kd_dataset(results), f, indent=2)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("gpt2-medium")
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")

def tokenize(example):
    prompt = example["input"]
    target = example["output"]
    tokenized_input = tokenizer(prompt, truncation=True, padding="max_length", max_length=512)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(target, truncation=True, padding="max_length", max_length=128)
    tokenized_input["labels"] = labels["input_ids"]
    return tokenized_input

from datasets import load_dataset
dataset = load_dataset("json", data_files="kd_data.json")["train"]
dataset = dataset.map(tokenize)

args = TrainingArguments(
    output_dir="./student-kd-checkpoints",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=5e-5,
    fp16=True,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    tokenizer=tokenizer
)

trainer.train()


In [ ]:
from transformers import pipeline
student_model = AutoModelForCausalLM.from_pretrained("./student-kd-checkpoints")
student_pipe = pipeline("text-generation", model=student_model, tokenizer=tokenizer)

# Run feedback generation or refinement with updated student

## Experiments with Dummy Models
We haven't integrated with the real models that we'll use YET. Crucially some of that code has been written but commented out for the time being.

**Fine Tuning Student Model Using SFT**

**Evaluate Distilled Student**

Imports and Stuff

In [ ]:
!pip install transformers
!pip uninstall datasets
!pip install datasets
!pip install transformers datasets accelerate sentence-transformers --quiet
#!pip install openai --quiet

Hugging Face Login

In [ ]:
!huggingface-cli login

More Imports and Setup

In [ ]:
import torch
import random
import re
import csv
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



# Load GSM8K subset for testing
gsm8k = load_dataset("gsm8k", "main", split="train[:5%]")

# Embedder for feedback scoring
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Mistral LLM (Hugging Face)
model_id = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Generator wrapper for reuse
def generate_llm(prompt, temp=0.3, max_tokens=256):
    return llm(prompt, max_new_tokens=max_tokens, temperature=temp, do_sample=True)[0]['generated_text']


# Set up OpenAI API key
#openai.api_key = "sk-..."   Replace with our actual key once thats ready

# Load a small GSM8K slice for fast testing -- only using small part of daaset
#gsm8k = load_dataset("gsm8k", "main", split="train[:1%]")

# Embedding model for cosine similarity
#embedder = SentenceTransformer("all-MiniLM-L6-v2")






Dummy Model Functions

In [ ]:
# Student model's initial answer
def base_model(prompt):
    full_prompt = f"{prompt}\n\nAnswer the question step by step. Conclude with '#### [final answer]'."
    return generate_llm(full_prompt)

# Expert-level feedback
def expert_feedback(prompt, x0):
    fb_prompt = f"""You are an expert tutor reviewing a student's math answer.

Question: {prompt}

Student's Answer:
{x0}

Provide helpful feedback to improve the answer."""
    return generate_llm(fb_prompt)

# Student self-critique
def student_feedback(prompt, x0):
    fb_prompt = f"""You are a student trying to critique your own math answer.

Question:
{prompt}

Your Answer:
{x0}

What might be wrong? Suggest how to improve it."""
    return generate_llm(fb_prompt)

# Refine answer based on both feedbacks
def refine_answer(prompt, x0, p_stu, p_exp):
    ref_prompt = f"""Refine this student's answer based on expert and self feedback.

Question: {prompt}

Original Answer:
{x0}

Expert Feedback:
{p_exp}

Student Feedback:
{p_stu}

Provide a better answer. End with '#### [answer]'."""
    return generate_llm(ref_prompt)



##THE REST IS WHAT THE ACTUAL MODEL WILL HAVE
# def base_model(prompt):
  #  system_msg = "You are a helpful math tutor. Solve the question step-by-step. Format your final answer with '#### [answer]' at the end."
   # user_msg = prompt

    #response = openai.ChatCompletion.create(
    #model="gpt-3.5-turbo",
    #messages=[
        #    {"role": "system", "content": system_msg},
         #   {"role": "user", "content": user_msg}
        #],
    #temperature=0.3,
    #max_tokens=512
    #)

    # return response["choices"][0]["message"]["content"]



# def expert_feedback(prompt, x0):
  #  system_msg = "You are an expert tutor. Your job is to critique student answers constructively and point out mistakes in a helpful way."
   # user_msg = f"""Here is the original question:

   # {prompt}

    #Here is the student's initial answer:

    #{x0}

    #Please provide:
    #1. A clarification question
    #2. A concrete suggestion to improve the student's answer
    #"""

    #response = openai.ChatCompletion.create(
        #model="gpt-3.5-turbo",
       # messages=[
      #      {"role": "system", "content": system_msg},
     #       {"role": "user", "content": user_msg}
    #    ],
   #     temperature=0.5,
  #      max_tokens=256
 #   )
#    return response["choices"][0]["message"]["content"]



#def student_feedback(prompt, x0):
 #   system_msg = "You are a student trying to reflect on your own math answer and improve it."
  #  user_msg = f"""Reflect on your solution to the following math problem:

#Question:
#{prompt}

#Your Answer:
#{x0}

#What might you have done wrong? Suggest one way to improve your answer."""

 #   response = openai.ChatCompletion.create(
  #      model="gpt-3.5-turbo",
   #     messages=[
    #        {"role": "system", "content": system_msg},
     #       {"role": "user", "content": user_msg}
      #  ],
       # temperature=0.7,
        #max_tokens=256
    #)
#    return response["choices"][0]["message"]["content"]

Utilities

In [ ]:

import re
from sklearn.metrics.pairwise import cosine_similarity

# 3.1 Cosine similarity scoring between student and expert feedback
def compute_similarity(fb1, fb2):
    vec1 = embedder.encode([fb1])
    vec2 = embedder.encode([fb2])
    return float(cosine_similarity(vec1, vec2)[0][0])

# 3.2 Extract first number from generated string (used for model outputs)
def extract_answer(text):
    for token in text.split():
        if token.isdigit():
            return int(token)
    return None

# 3.3 Extract final answer from GSM8K-style annotated solutions (e.g., '#### 42')
def extract_final_answer(text):
    match = re.search(r"####\s*(-?\d+)", text)
    return int(match.group(1)) if match else None

# 3.4 Save function (adapted from Henry's main code — used by Aayush)
def save_results_to_csv(result, features, file_path):
    import csv  # local import ensures it's always available
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        row = [result.get(feature, "") for feature in features]
        writer.writerow(row)


Main Experiment Loop

In [ ]:
max_iters = 1  # Will support iterative refinement later
threshold = 0.01  # Placeholder for similarity-based convergence

results = []

for i, example in enumerate(gsm8k):
    input_prompt = example["question"]
    ground_truth = extract_final_answer(example["answer"])

    # 🔍 Skip examples with invalid or missing final answers
    if ground_truth is None:
        print(f"⚠️ Could not extract answer from:\n{example['answer']}")
        continue

    print(f"\n--- Task {i+1} ---")
    print(f"Prompt: {input_prompt}")

    # Step 1: Generate initial output
    x0 = base_model(input_prompt)
    print("Initial Output:", x0.strip())

    # Step 2: Expert critique
    p_exp = expert_feedback(input_prompt, x0)
    print("Expert Feedback:", p_exp.strip())

    # Step 3: Student self-critique
    p_stu = student_feedback(input_prompt, x0)
    print("Student Feedback:", p_stu.strip())

    # Step 4: Feedback similarity scoring
    score = compute_similarity(p_exp, p_stu)
    print("Feedback Similarity:", round(score, 3))

    # Step 5: Generate refined output using feedback
    x1 = refine_answer(input_prompt, x0, p_stu, p_exp)
    print("Refined Output:", x1.strip())

    # Step 6: Evaluate prediction
    pred_answer = extract_final_answer(x1)
    accuracy = int(pred_answer == ground_truth) if pred_answer is not None else 0
    print("Prediction:", pred_answer, "| Ground Truth:", ground_truth)
    print("Correct:", bool(accuracy))

    # Step 7: Log result to list + file
    result = {
        "input": input_prompt,
        "x0": x0,
        "p_expert": p_exp,
        "p_student": p_stu,
        "similarity": score,
        "x1": x1,
        "pred": pred_answer,
        "label": ground_truth,
        "correct": bool(accuracy)
    }

    results.append(result)

    save_results_to_csv(
        result=result,
        features=["input", "x0", "p_expert", "p_student", "similarity", "x1", "pred", "label", "correct"],
        file_path="gsm8k_self_critique_results.csv"
    )

In [ ]:
import json

file_path = "1k_dataset.jsonl"
expert_datasets = []

with open(file_path, "r") as f:
    for line in f:
        item = json.loads(line)

        prompt = item["prompt"]
        original_answer = item["original_answer"]
        score = item["score"]
        feedback = item["feedback"]
        expert_datasets.append(
            {"prompt": prompt,
             "answer": original_answer,
             "feedback": feedback,
             "score": score}
        )

print(f" {len(expert_datasets)} examples loaded")

## Unified model

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from bert_score import score as bert_score
import re

class SelfImprovingModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", network=None, model_path=None, lr=5e-5, device=None, penalty_factor=1.0):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path if model_path else model_name,
            torch_dtype=self.model_dtype,
            device_map="auto"
        )

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=lr)
        self.last_score = 0.0
        self.penalty_factor = penalty_factor
        self.feedback_prompt_condenser = GPTPromptFeedbackGenerator()
        self.network = network

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate_prompt(self, question, previous_answer=None, unified_feedback=None, cot=None):
        prompt = ""

        if cot:
            prompt += cot.replace("{replace with the prompt}", question.strip() + "\n\n")
        else:
            prompt += f"Q: {question.strip()}\n"

        if previous_answer:
            if isinstance(previous_answer, tuple):
                previous_answer = previous_answer[1]
            prompt += f"\nPrevious Answer:\n{previous_answer.strip()}\n"

            if self.last_score:
                prompt += f"\nThe previous answer received a score of {self.last_score} on a scale from -1.0 to 1.0."

            if unified_feedback:
                prompt += f"\n\nFeedback:\n{unified_feedback.strip()}"
            else:
                prompt += "\n\n(Feedback not available yet. Try to improve the answer based on clarity, accuracy, and completeness.)"

            prompt += (
                "\n\nYour task is to generate an improved answer using the feedback and prior response as reference."
                "\n- Focus on clarity, accuracy, and logical completeness."
                "\n- Build upon strengths of the previous answer."
                "\n- Ensure the response is coherent and self-contained.\n"
                "\nAnswer:\nA:"
            )
        else:
            # No previous answer – generate from scratch
            prompt += (
                "\nThere is no previous answer available.\n"
                "Please generate a new answer to the question above."
                "\n- Aim for clarity, accuracy, and completeness."
                "\n- Ensure the answer is logically structured and self-contained.\n"
                "\nAnswer:\nA:"
            )

        return prompt



    def generate_answer(self, question, previous_answer=None, previous_score=0.0, previous_feedback=None, cot=None, max_new_tokens=100):
        self.last_score = previous_score

        # Step 1: If previous_answer exists, generate feedback
        if self.network and previous_answer is not None:
            initial_feedback = self.network.generate_combined_feedback(question, previous_answer)
            expert_feedback = initial_feedback.get("expert_feedback")
            amateur_feedback = initial_feedback.get("amateur_feedback")
            unified_feedback = self.feedback_prompt_condenser.generate_unified_feedback(
                expert_feedback, amateur_feedback, self.last_score
            )
        else:
            unified_feedback = previous_feedback or "No feedback provided."

        # Step 2: Construct prompt for revision using unified feedback
        prompt = self.generate_prompt(
            question=question,
            previous_answer=previous_answer,
            unified_feedback=unified_feedback,
            cot=cot
        )

        # Step 3: Generate a revised answer
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        output = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=1.0,
            do_sample=False
        )

        generated_tokens = output[0][inputs["input_ids"].shape[-1]:]  # Only decode the generated continuation
        revised_answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


        # Step 4: Generate feedback for the revised answer
        final_feedback = self.network.generate_combined_feedback(question, revised_answer)
        expert_feedback = final_feedback.get("expert_feedback")
        amateur_feedback = final_feedback.get("amateur_feedback")

        # Step 5: Condense feedback again based on improved score
        improved_score = final_feedback.get("combined_score", 0.0)
        unified_feedback = self.feedback_prompt_condenser.generate_unified_feedback(
            expert_feedback, amateur_feedback, improved_score
        )

        return unified_feedback, revised_answer, improved_score



    def prepare_inputs_and_labels(self, prompt, answer, score, feedback):
        full_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
            f"Score: {score}\n"
            f"Feedback: {feedback}\n"
        )

        tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096)
        input_ids = tokenized.input_ids[0].clone()
        labels = input_ids.clone()

        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        score_token_ids = torch.tensor(score_token_ids, device=input_ids.device)

        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i + len(score_token_ids)], score_token_ids):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0

        labels[:start_idx] = -100

        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            tokenized.attention_mask.to(self.device)
        )

    def generate_feedback(self, prompt, answer, max_new_tokens=150):
        self.model.eval()

        input_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            "Respond exactly in this format:\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        pattern = r"Score:\s*([-+]?\d*\.?\d+)\s*Feedback:\s*(.*)"
        match = re.search(pattern, generated_text, re.DOTALL)

        if match:
            score = match.group(1).strip()
            feedback = match.group(2).strip().split("\n")[0]
            return f"Score: {score}\nFeedback: {feedback}"
        else:
            return generated_text.strip()

    def fine_tuning(self, train_dataset, output_dir, stopped=100, epochs=1, print_every=10, lm_loss_weight=0.2, semantic_loss_weight=0.8):
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0.0

            for i, sample in enumerate(train_dataset):
                if i == stopped:
                    break

                prompt = sample.get("prompt", "")
                answer = sample.get("original_answer", "")
                score = sample.get("score", "")
                feedback = sample.get("feedback", "")

                input_ids, labels, attention_mask = self.prepare_inputs_and_labels(prompt, answer, score, feedback)

                outputs = self.model(
                    input_ids=input_ids,
                    labels=labels,
                    attention_mask=attention_mask
                )
                lm_loss = outputs.loss

                predicted_feedback = self.generate_feedback(prompt, answer)
                P, R, F1 = bert_score([predicted_feedback], [feedback], lang="en", verbose=False)
                semantic_loss = 1.0 - F1[0].item()
                semantic_loss = torch.tensor(semantic_loss, dtype=self.model_dtype, device=self.device)

                total_combined_loss = lm_loss_weight * lm_loss + semantic_loss_weight * semantic_loss

                self.optimizer.zero_grad()
                total_combined_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += total_combined_loss.item()

                if (i + 1) % print_every == 0:
                    print(f"\n--- Sample {i+1} (Epoch {epoch+1}) ---")
                    print(f"Prompt: {prompt}")
                    print(f"Answer: {answer}")
                    print(f"GT Score: {score}")
                    print(f"GT Feedback:\n{feedback}")
                    print(f"\nGenerated Feedback:\n{predicted_feedback}")
                    print(f"LM Loss: {lm_loss.item():.4f}")
                    print(f"Semantic Loss: {semantic_loss.item():.4f}")
                    print(f"Total Loss: {total_combined_loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / (i + 1)
            print(f"\nEpoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}\n")

        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)

    def update_model_with_feedback(self, question, previous_answer, previous_feedback, cot=None):
        combined_feedback, improved_answer, improved_score = self.generate_answer(
            question, previous_answer, previous_feedback, cot
        )

        prev_score = self.network.generate_combined_feedback(question, previous_answer)["combined_score"]
        delta = improved_score - prev_score
        reward = delta if delta >= 0 else delta * self.penalty_factor

        min_reward_magnitude = 0.05
        if abs(reward) < min_reward_magnitude and reward != 0:
            reward = reward / abs(reward) * min_reward_magnitude

        is_satisfied, critique = self.feedback_prompt_condenser.check_feedback_satisfaction(
            question, previous_answer, combined_feedback, improved_answer
        )
        reward += 0.2 if is_satisfied else -0.2

        print(f"Previous Score: {prev_score:.4f}, Improved Score: {improved_score:.4f}, Reward: {reward:.4f}")
        print(f"Critique: {critique}\n")

        return improved_answer, combined_feedback, improved_score, reward

    def self_critique(self, prompt, previous_answer, previous_feedback, cot=None, iterations=10, target_score=0.95):
        best_answer = previous_answer
        feedback = previous_feedback
        best_score = self.network.generate_combined_feedback(prompt, previous_answer)["combined_score"]

        previous_answers = set()
        previous_answers.add(best_answer)

        for step in range(iterations):
            if best_score >= target_score:
                print(f"Target score reached ({best_score:.4f}), stopping early.")
                break

            improved_answer, feedback, improved_score, reward = self.update_model_with_feedback(
                prompt, best_answer, feedback, cot
            )

            if improved_answer in previous_answers:
                reward -= 0.1
                print("Penalty: Repeated answer detected.")

            previous_answers.add(improved_answer)

            print(f"Step {step}: Reward {reward:.4f}, Score {improved_score:.4f}")
            print(f"Improved Answer:\n{improved_answer}\n")

            if improved_score > best_score:
                best_answer = improved_answer
                best_score = improved_score

        print(f"Final Answer:\n{best_answer}")
        print(f"Final Score: {best_score:.4f}")
        return best_answer, best_score

    def fine_tuning_text_generation(self, dataset, output_dir, cot = None, stopped=100, epochs=1, print_every=10):
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0.0

            for i, sample in enumerate(dataset):
                if i == stopped:
                    break

                question = sample.get("question")
                previous_answer = sample.get("original_answer", None)
                improved_answer = sample.get("improved_answer")
                feedback = sample.get("feedback", None)

                full_prompt = self.generate_prompt(
                    question=question,
                    previous_answer=previous_answer,
                    unified_feedback=feedback,
                    cot=cot,
                    include_answer_tag=True
                )
                full_text = full_prompt + " " + improved_answer.strip()

                tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096)
                input_ids = tokenized.input_ids[0]
                attention_mask = tokenized.attention_mask[0]

                prompt_ids = self.tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=4096).input_ids[0]
                labels = input_ids.clone()
                labels[:len(prompt_ids)] = -100  # Ignore prompt tokens in loss

                # Send to device
                input_ids = input_ids.unsqueeze(0).to(self.device)
                attention_mask = attention_mask.unsqueeze(0).to(self.device)
                labels = labels.unsqueeze(0).to(self.device)

                # Forward + loss
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

                # Backward pass
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += loss.item()

                if (i + 1) % print_every == 0:
                    print(f"\n--- Sample {i + 1} (Epoch {epoch + 1}) ---")
                    print(f"Prompt:\n{full_prompt}")
                    print(f"\nImproved Answer:\n{improved_answer}")
                    print(f"Loss: {loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / (i + 1)
            print(f"\nEpoch {epoch + 1} completed - Avg Loss: {avg_loss:.4f}\n")

        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Initial Answer: Why is the sky blue?

1. The sun is the primary source of light in the sky.

2. The Earth's atmosphere absorbs and reflects some of the sun's light.

3. The Earth's rotation around the sun causes the sun's rays to travel in a circular path around the Earth.

4. The Earth's rotation causes the sun's rays to travel in a zigzag pattern, which creates the appearance of a blue
--- Iteration 1 ---
Feedback: Prompt: Why is the sky blue?
Answer: Why is the sky blue?

1. The sun is the primary source of light in the sky.

2. The Earth's atmosphere absorbs and reflects some of the sun's light.

3. The Earth's rotation around the sun causes the sun's rays to travel in a circular path around the Earth.

4. The Earth's rotation causes the sun's rays to travel in a zigzag pattern, which creates the appearance of a blue
Give constructive feedback:

1. "Great job! Your explanation is clear and concise. Can you add some information about how the Earth's atmosphere affects the color of t

KeyboardInterrupt: 

In [ ]:
import openai

openai.api_key = "sk-xxxxxxxx"

class GPTPromptFeedbackGenerator:
    def __init__(self, model_name = "gpt-4o"):
        self.model_name = model_name

    def prompt_condense_feedback(self, expert_feedback, amateur_feedback, last_score):
        prompt = (
            f"The previous answer received a score of {last_score} (scale: -1.0 to 1.0), where:\n"
            "-1.0 = completely incorrect or misleading\n"
            " 0.0 = partially helpful or neutral\n"
            " 1.0 = accurate, complete, and helpful\n\n"
            "You are given two sets of feedback on this answer:\n"
            f"- Expert Feedback (more reliable): {expert_feedback.strip()}\n"
            f"- Amateur Feedback: {amateur_feedback.strip()}\n\n"
            "Your task is to **condense** these feedbacks into a single, unified summary that:\n"
            "- Prioritizes the expert feedback\n"
            "- Includes any valid or helpful insights from the amateur feedback\n"
            "- Avoids redundancy or contradiction\n"
            "- Clearly communicates how the answer could be improved (in clarity, completeness, structure, logic, etc.)\n\n"
            "**Final Output:** A concise, constructive, and actionable feedback summary for the answer.\n\n"
            "Unified Feedback:"
        )
        return prompt


    def generate_unified_feedback(self, expert_feedback = None, amateur_feedback = None, last_score = None):

        if not expert_feedback and not amateur_feedback:
            return "No feedback available yet."

        full_prompt = self.prompt_condense_feedback(expert_feedback, amateur_feedback, last_score)

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that condenses expert and amateur feedback into a unified improved answer."},
                    {"role": "user", "content": full_prompt}
                ],
                temperature=0.7,
                max_tokens=512
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Error generating feedback: {e}")
            return None

    def check_feedback_satisfaction(self, question, previous_answer, feedback, current_answer):
        review_prompt = (
            f"Question: {question}\n\n"
            f"Previous Answer: {previous_answer}\n\n"
            f"Combined Feedback (from expert and amateur evaluators): {feedback}\n\n"
            f"Current Answer: {current_answer}\n\n"
            "Based on the combined feedback, does the current answer appropriately address the most important points raised?\n"
            "Consider both expert and amateur feedback but prioritize the expert perspective. "
            "Your reply should begin with 'Yes' or 'No'. If 'No', briefly explain what is still missing or unclear."
        )

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a critical reviewer assessing if feedback was properly incorporated."},
                    {"role": "user", "content": review_prompt}
                ],
                temperature=0.5,
                max_tokens=150,
            )

            output = response.choices[0].message.content.strip()
            print(f"Feedback Satisfaction Check:\n{output}\n")
            return output.lower().startswith("yes"), output

        except Exception as e:
            print(f"OpenAI API error: {e}")
            return False, "API Error"

In [ ]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

class SelfImprovingModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", network = None, device = None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        self.network = network
        self.gpt_prompt_condenser = GPTPromptFeedbackGenerator()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    def generate_text(self, prompt, max_new_tokens=300):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.3,
            top_p=0.7,
            pad_token_id=self.tokenizer.eos_token_id
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, max_new_tokens=300, max_words = 100):
        prompt = f"""You are a helpful academic assistant.
    Answer the following question clearly and accurately.
    Limit your answer to approximately {max_words} words.
    Do not generate multiple Q&A.

    Question: {question}

    Answer:"""

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens)

        # Extract only the portion after "Answer:"
        if "Answer:" in response:
            answer = response.split("Answer:", 1)[1].strip()
        else:
            answer = response.strip()

        return answer


    def generate_self_critique(self, question, answer, target_score=1.0):
        prompt = (
            "You are a critical writing tutor. Analyze the following answer for logic, clarity, accuracy, and depth.\n"
            f"Your goal is to critique the answer constructively so it can be improved to reach a quality score of approximately {target_score:.1f} on a scale from -1 (very poor) to 1 (excellent).\n\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n"
            "Critique (focus on how to improve to reach the target score):"
        )

        response = self.generate_text(prompt, max_new_tokens=300)

        # Extract the part after "Critique:"
        marker = "Critique (focus on how to improve to reach the target score):"
        if marker in response:
            critique = response.split(marker, 1)[1].strip()
        else:
            critique = response.strip()

        return critique

    def generate_feedback(self, question, answer):
        prompt = (
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
            "You are an expert evaluator. Assess the overall quality of the answer in relation to the question.\n"
            "Assign a single score strictly between -1.0 and 1.0 based on the rubric below:\n"
            "• 1.0 – Excellent: Fully accurate, comprehensive, insightful, and well-structured.\n"
            "• 0.7–0.9 – Good: Mostly accurate and valuable, with only minor issues or omissions.\n"
            "• 0.4–0.6 – Fair: Some helpful content, but includes notable flaws, confusion, or missing key points.\n"
            "• 0.1–0.3 – Weak: Largely inaccurate, unclear, or oversimplified; lacks depth or structure.\n"
            "• 0.0 – Unhelpful: Irrelevant or does not meaningfully respond to the question.\n"
            "• -0.1 to -1.0 – Misleading or harmful: Contains serious factual errors, false claims, or dangerous reasoning.\n\n"
            "Then, write a **detailed paragraph** that explains *why* you gave this score.\n"
            "Do not just say 'revise' or 'clarify'. Instead:\n"
            "• Identify specific strengths or weaknesses.\n"
            "• Highlight examples of correct or incorrect reasoning.\n"
            "• Suggest what is missing or could be improved.\n"
            "• Avoid vague phrases like 'lacks clarity' or 'needs revision' without elaboration.\n\n"
            "Respond in this exact format:\n"
            "Score <how do you rate the answer from -1 to 1?>\n\n"
            "Feedback: <1-paragraph explanation of the score with detailed reasoning and examples>"
        )


        response = self.generate_text(prompt, max_new_tokens=300)

        # Extract score
        match = re.search(r"Score\s*:\s*(-?\d+(?:\.\d+)?)", response)
        score = float(match.group(1)) if match else 0.0

        feedback_matches = re.findall(r"Feedback\s*:\s*(.*?)(?:\n\n|\Z)", response, re.DOTALL)

        # Take the last one (likely to be the actual feedback)
        feedback = feedback_matches[-1].strip() if feedback_matches else "No feedback provided."

        # Clean up "Example:" label if present
        feedback = re.sub(r"(?i)^example\s*:\s*", "", feedback).strip()

        return feedback, score

    def apply_feedback(self, question, answer, feedback):
        prompt = (
            "Revise the following answer to the question using the feedback provided. "
            "Ensure the revised answer is clear, logical, and well-supported.\n\n"
            f"Question: {question}\n"
            f"Original Answer: {answer}\n"
            f"Feedback: {feedback}\n"
            "Improved Answer:"
        )

        response = self.generate_text(prompt, max_new_tokens=300)

        # Extract everything after "Improved Answer:"
        split_key = "Improved Answer:"
        if split_key in response:
            improved_answer = response.split(split_key, 1)[1].strip()
        else:
            improved_answer = response.strip()

        return improved_answer

    def improve_answer_with_feedback_and_critique(self, question, answer, target_score=1.0):
        # Step 1: Generate feedback and score
        original_feedback_dict = self.network.generate_combined_feedback(question, answer)

        original_combined_feedback_expert = original_feedback_dict["expert_feedback"]
        original_combined_feedback_amateur = original_feedback_dict["amateur_feedback"]

        original_score = original_feedback_dict["combined_score"]

        original_condensed_feedback = self.gpt_prompt_condenser.generate_unified_feedback(original_combined_feedback_expert, original_combined_feedback_amateur)

        try:
            score = float(original_score)
        except ValueError:
            print(f"Warning: Could not convert score '{score}' to float.")
            score = -1.0  # Treat as poor response to trigger improvement

        if score >= target_score:
            return answer.strip(), score, False  # No need to improve

        # Step 2: Apply feedback to improve the answer
        improved_answer = self.apply_feedback(question, answer, original_condensed_feedback)

        # Step 3: Generate critique based on improved answer
        critique = self.generate_self_critique(question, improved_answer)

        # Step 4: Refine again using critique
        refined_answer = self.apply_feedback(question, improved_answer, critique)

        # Step 5: Get final score
        final_feedback_dict = self.network.generate_combined_feedback(question, refined_answer)

        final_combined_feedback_expert = final_feedback_dict["expert_feedback"]
        final_combined_feedback_amateur = final_feedback_dict["amateur_feedback"]
        final_score = final_feedback_dict["combined_score"]

        final_condensed_feedback = self.gpt_prompt_condenser.generate_unified_feedback(final_combined_feedback_expert, final_combined_feedback_amateur)

        try:
            final_score = float(final_score)
        except ValueError:
            print(f"Warning: Could not convert final score '{final_score}' to float.")
            final_score = -1.0
        return refined_answer.strip(), final_score, True

    def prepare_inputs_and_labels(self, prompt, answer, expert_score=None, expert_feedback=None):
        full_prompt = f"Question: {prompt}\nAnswer: {answer}"
        inputs = self.tokenizer(full_prompt, return_tensors="pt", truncation=True, padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        labels = input_ids.clone()
        return input_ids, labels, attention_mask

In [ ]:
import openai
import re

class GPTExpertFeedbackModel(BaseMockFeedbackModel):
    def __init__(self, model_name="gpt-4o"):
        super().__init__(name="expert")
        self.model_name = model_name

    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert evaluator. Your task is to score the quality of the following answer based "
                "on how well it responds to the given prompt.\n\n"
                "Use the following detailed scoring scale:\n"
                "1.0: Perfect completion with no issues.\n"
                "0.5 – 0.9: Generally good, minor flaws.\n"
                "0.1 – 0.4: Partially helpful, moderate issues.\n"
                "0.0: Neutral or off-topic, neither helpful nor harmful.\n"
                "-0.1 – -0.4: Mildly harmful or misleading.\n"
                "-0.5 – -0.9: Significantly off-task or misleading.\n"
                "-1.0: Complete failure or harmful content.\n\n"
                "Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.\n"
                "Respond exactly in this format:\n"
                "[reason]xxxx (MAX 50 words). Score: [score]\n\n"
            )
        }

        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0,
                max_tokens=150,
                n=1
            )
            output = response.choices[0].message.content.strip()
            print(f"[Expert Model Feedback]\n{output}\n")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", output, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract feedback (text before "Score")
            feedback = output.split("Score")[0].strip()

            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Error parsing GPT expert feedback: {e}"

        return score, feedback


In [ ]:
import openai
import re

class GPTAmateurFeedbackModel(BaseMockFeedbackModel):
    def __init__(self, model_name="gpt-3.5-turbo"):
        super().__init__(name="amateur")
        self.model_name = model_name

    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an amateur evaluator learning how to assess answers. Your job is to give simple feedback "
                "and assign a rough score based on how helpful or understandable the answer is.\n\n"
                "Use this scoring scale:\n"
                "1.0: Great answer, easy to understand.\n"
                "0.5 – 0.9: Pretty good, some things could be clearer.\n"
                "0.1 – 0.4: Okay, but a bit confusing or missing things.\n"
                "0.0 or below: Not helpful, or doesn't make sense.\n\n"
                "Try your best to explain in everyday language what worked and what didn't.\n"
                "Respond in this format:\n"
                "[short feedback] Score: [score]\n\n"
            )
        }

        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0.7,
                max_tokens=150
            )

            output = response.choices[0].message.content.strip()
            print(f"[Amateur Model Feedback]\n{output}\n")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", output, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract feedback
            feedback = output.split("Score")[0].strip()

            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Error parsing GPT amateur feedback: {str(e)}"

        return score, feedback

In [ ]:
class MockAmateurExpertFeedbackNetWork:
    def __init__(self):
        self.teacher = GPTExpertFeedbackModel()
        self.student = GPTAmateurFeedbackModel()

    def generate_combined_feedback(self, prompt_text, answer, alpha=0.8):
        # Get individual scores and feedback
        student_score, student_feedback = self.student.predict_score_and_feedback(prompt_text, answer)
        teacher_score, teacher_feedback = self.teacher.predict_score_and_feedback(prompt_text, answer)

        # Weighted average score favoring expert
        combined_score = alpha * teacher_score + (1 - alpha) * student_score

        # Combined text (for revision prompt)
        combined_feedback = (
            f"The expert highlights: {teacher_feedback}\n"
            f"The amateur also adds: {student_feedback}"
        )

        return {
            "combined_score": combined_score,
            "combined_feedback": combined_feedback,
            "expert_feedback": teacher_feedback,
            "amateur_feedback": student_feedback
        }

In [ ]:
if __name__ == "__main__":
    dummy_network = MockAmateurFeedbackNetwork()
    model = SelfImprovingModel(network=dummy_network)

    question = "Why do you want to become a software engineer?"
    previous_answer = "I enjoy solving problems and building things that can help people."
    previous_feedback = "This is a good start, but it's a bit generic. Try adding a personal story or specific goals to make it more compelling."


    print("\n=== Initial Answer ===")
    print(previous_answer)

    print("\n=== Starting Self-Critique ===")
    final_answer, final_score = model.self_critique(
        prompt=question,
        previous_answer=previous_answer,
        previous_feedback=previous_feedback,
        iterations=10,
        target_score=0.9
    )

    print("\n=== Final Answer ===")
    print(final_answer)
    print(f"Final Score: {final_score:.4f}")



=== Initial Answer ===
I enjoy solving problems and building things that can help people.

=== Starting Self-Critique ===
Previous Score: 0.3000, Improved Score: 0.0000, Reward: -0.5000
Critique: Still missing thermodynamic context.

Step 0: Reward -0.5000, Score 0.0000
Improved Answer:
A: I am passionate about solving complex problems and building things that can help people. I have always been fascinated by the way that technology can transform the way we live, work, and communicate.

As a software engineer, I aim to create software that is not only functional but also beautiful, intuitive, and user-friendly. I am particularly interested in developing software that can help people in their daily lives, such as scheduling apps, productivity tools, and social media platforms

Previous Score: 0.3000, Improved Score: 0.0000, Reward: -0.5000
Critique: Still missing thermodynamic context.

Step 1: Reward -0.5000, Score 0.0000
Improved Answer:
A: I am passionate about solving complex probl

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from bert_score import score as bert_score  # optional
except ImportError:
    bert_score = None


class SelfImprovingModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", network=None, device=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)

        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel.")
            self.model = torch.nn.DataParallel(self.model)

        self.model.to(self.device)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model_dtype = next(self.model.parameters()).dtype

        self.network = network
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=2e-5)

    def generate_text(self, prompt, max_new_tokens=300, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        model = self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=0.5,
            pad_token_id=self.tokenizer.eos_token_id
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, max_new_tokens=150, max_words=100):
        prompt = (
            "You are a helpful academic assistant.\n"
            "Answer the following question clearly and accurately.\n"
            f"Limit your answer to approximately {max_words} words.\n"
            "Do not generate multiple Q&A.\n\n"
            f"Question: {question}\n\nAnswer:"
        )
        response = self.generate_text(prompt, max_new_tokens)
        return response.split("Answer:", 1)[-1].strip()

    def generate_self_critique(self, question, answer):
        prompt = (
            "You are an expert evaluator. Write a concise, 50-word critique explaining why the revised answer is an improvement.\n"
            "Focus on clarity, motivation, tone, and detail. Avoid repeating phrases like 'more effective.' Use varied language and be specific.\n\n"
            f"Question:\n{question}\n\nRevised Answer:\n{answer}\n\nCritique (exactly 50 words):"
        )
        response = self.generate_text(prompt, max_new_tokens=150)
        return response.split("Critique (exactly 50 words):", 1)[-1].strip()

    def generate_feedback(self, question, answer):
        prompt = (
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
            "Evaluate the quality and relevance of the answer in response to the question. "
            "Rate it using the scale below and provide a brief justification for your score:\n\n"
            "  •  1.0 — Excellent: completely correct, clear, and highly relevant\n"
            "  •  0.7 to 0.9 — Good: mostly correct and relevant, with minor issues\n"
            "  •  0.4 to 0.6 — Fair: some useful elements, but has clear problems\n"
            "  •  0.1 to 0.3 — Weak: limited value, vague or largely off-topic\n"
            "  •  0.0 — Not helpful: off-topic, irrelevant, or no useful information\n"
            "  • -0.1 to -0.4 — Misleading: somewhat incorrect or confusing\n"
            "  • -0.5 to -0.9 — Wrong: mostly incorrect or misleading\n"
            "  • -1.0 — Harmful: completely wrong or potentially dangerous\n\n"
            "Respond only in this format:\n"
            "Score: <number>\n"
            "Feedback: <brief justification for the score>"
        )

        response = self.generate_text(prompt, max_new_tokens=150).strip()

        # Extract all score values
        score_matches = re.findall(r"Score\s*:\s*(-?1(?:\.0)?|-?0(?:\.\d+)?|-\d\.\d+)", response)
        score = float(score_matches[0]) if score_matches else 0.0

        # Extract all feedback matches
        feedback_matches = re.findall(r"Feedback\s*:\s*(.*?)(?=\n\S|$)", response, re.DOTALL)

        if not feedback_matches:
            feedback = "No feedback provided."
        else:
            # Choose the middle feedback if there are multiple, otherwise use the only one
            mid_idx = len(feedback_matches) // 2
            feedback = feedback_matches[mid_idx].strip()

        return feedback, score

    def apply_feedback(self, question, answer, feedback):
        prompt = (
            "You are an expert writing assistant. Revise the following answer using the feedback provided. "
            "Be concise, logically structured, and focused. Do not add commentary or extra remarks.\n\n"
            f"Question: {question}\n"
            f"Original Answer: {answer}\n"
            f"Feedback: {feedback}\n\n"
            "Only output the improved answer below:\nImproved Answer:"
        )
        response = self.generate_text(prompt)
        return response.rsplit("Improved Answer:", 1)[-1].strip()

    def improve_answer_with_feedback_and_critique(self, question, answer, target_score=1.0):
        original_feedback_dict = self.network.generate_combined_feedback(question, answer)

        expert_feedback = original_feedback_dict.get("expert_feedback", "")
        amateur_feedback = original_feedback_dict.get("amateur_feedback", "")
        combined_score = original_feedback_dict.get("combined_score", -1.0)

        # Get merged feedback from expert model
        condensed_feedback = expert_model.generate_unified_feedback(expert_feedback, amateur_feedback)
        first_paragraph = condensed_feedback.strip().split('\n\n')[0].strip()


        try:
            score = float(combined_score)
        except (ValueError, TypeError):
            score = -1.0

        if score >= target_score:
            return answer.strip(), score, False

        improved = self.apply_feedback(question, answer, first_paragraph)
        critique = self.generate_self_critique(question, improved)
        refined = self.apply_feedback(question, improved, critique)

        final_feedback_dict = self.network.generate_combined_feedback(question, refined, KD=False)
        final_score = float(final_feedback_dict.get("combined_score", -1.0))

        return refined.strip(), final_score, True

    def prepare_inputs_and_labels(self, prompt, answer, expert_score, expert_feedback):
        # Prepare the training sample string in a strict format
        score_str = str(expert_score)
        full_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n"
            "Respond in this exact format:\n\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
            f"Score: {score_str}\n"
            f"Feedback: {expert_feedback}\n"
        )

        # Tokenize the full input
        tokenized = self.tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        )
        input_ids = tokenized.input_ids[0]
        labels = input_ids.clone()

        # Locate the start index of the "Score:" token — the point from which loss is calculated
        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        score_token_ids = torch.tensor(score_token_ids, device=input_ids.device)

        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i + len(score_token_ids)], score_token_ids):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0  # fallback to compute loss from start

        # Mask out everything before "Score:" for loss calculation
        labels[:start_idx] = -100

        # Return input, labels, and attention mask — all moved to correct device
        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            tokenized.attention_mask.to(self.device)
        )

In [ ]:
self_improving_model = SelfImprovingModel()

question = "Why do you want to become a machine learning engineer?"
original_answer = "Because the ocean reflects its color into the sky."
feedback, score = self_improving_model.generate_feedback(question, original_answer)

In [ ]:
print(f"Feedback: {feedback}")
print(f"Score: {score}")

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

def clean_bracketed_notes(text):
    # Remove all [bracketed notes] (trailing or inline)
    return re.sub(r"\s*\[.*?\]", "", text).strip()

def truncate_words(text, max_words):
    words = text.split()
    return " ".join(words[:max_words]) if len(words) > max_words else text

class ExpertFeedbackModel:
    def __init__(self, model_name="elinas/Llama-3-13B-Instruct", network=None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel.")
            self.model = torch.nn.DataParallel(self.model)

        self.model.to(self.device)
        self.network = network

    def generate_text(self, prompt, max_new_tokens=300, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        model = self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=0.5,
            pad_token_id=self.tokenizer.eos_token_id
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, max_new_tokens=300, max_words=100):
        prompt = (
            "You are a helpful academic assistant.\n"
            f"Provide a clear and accurate answer to the question below in approximately {max_words} words.\n"
            "Respond with a single answer only—no multiple Q&As.\n\n"
            "When you have finished your sentence, please write [Stop]\n"
            f"Question: {question}\n\nAnswer:"
        )

        response = self.generate_text(prompt, max_new_tokens)
        answer = response.split("Answer:", 1)[-1].strip()

        # Truncate at "[Stop]" if it exists
        answer = answer.split("[Stop]", 1)[0].strip()

        # Remove any leftover bracketed notes
        answer = clean_bracketed_notes(answer)

        # Optionally truncate to max_words
        answer = truncate_words(answer, max_words)

        return answer


    def generate_feedback(self, question, answer):
        prompt = (
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
            "Evaluate the quality and relevance of the answer in response to the question. "
            "Rate it using the scale below and provide a brief justification for your score:\n\n"
            "  •  1.0 — Excellent: completely correct, clear, and highly relevant\n"
            "  •  0.7 to 0.9 — Good: mostly correct and relevant, with minor issues\n"
            "  •  0.4 to 0.6 — Fair: some useful elements, but has clear problems\n"
            "  •  0.1 to 0.3 — Weak: limited value, vague or largely off-topic\n"
            "  •  0.0 — Not helpful: off-topic, irrelevant, or no useful information\n"
            "  • -0.1 to -0.4 — Misleading: somewhat incorrect or confusing\n"
            "  • -0.5 to -0.9 — Wrong: mostly incorrect or misleading\n"
            "  • -1.0 — Harmful: completely wrong or potentially dangerous\n\n"
            "Respond only in this format:\n"
            "Score: <number>\n"
            "Feedback: <brief justification for the score>"
        )

        response = self.generate_text(prompt, max_new_tokens=200)

        # Extract score
        score_matches = re.findall(r"Score\s*:\s*(-?1(?:\.0+)?|-?0(?:\.\d+)?|-\d\.\d+)", response)
        score = float(score_matches[-1]) if score_matches else 0.0

        # Extract feedback
        feedback_matches = re.findall(r"Feedback\s*:\s*(.*?)(?=\n\S|$)", response, re.DOTALL)
        feedback = feedback_matches[-1].strip() if feedback_matches else "No feedback provided."
        feedback = re.sub(r"(?i)^example\s*:\s*", "", feedback).strip()

        return feedback, score

    def prompt_condense_feedback(self, expert_feedback, amateur_feedback):
        return (
            "You are given two feedback comments on an answer:\n"
            f"- Expert Feedback (trusted): {expert_feedback.strip()}\n"
            f"- Amateur Feedback: {amateur_feedback.strip()}\n\n"
            "Your task is to write a unified feedback summary (max 70 words) that:\n"
            "- Prioritizes insights from the expert feedback\n"
            "- Incorporates any unique and constructive points from the amateur feedback\n"
            "- Resolves disagreements by favoring the expert\n"
            "- Uses contrastive reasoning to emphasize the strengths of the expert feedback\n\n"
            "Final Merged Feedback:"
        )

    def generate_unified_feedback(self, expert_feedback, amateur_feedback):
        full_prompt = self.prompt_condense_feedback(expert_feedback, amateur_feedback)
        response = self.generate_text(full_prompt)

        if "Final Merged Feedback:" in response:
            return response.split("Final Merged Feedback:")[-1].strip()
        return response.strip()

    def apply_feedback(self, question, answer, feedback):
        prompt = (
            "You are an expert writing assistant. Revise the answer below based on the provided feedback. "
            "Ensure the revision is concise, logically organized, and focused. Do not include explanations or commentary.\n\n"
            f"Question: {question}\n"
            f"Original Answer: {answer}\n"
            f"Feedback: {feedback}\n\n"
            "Provide only the revised answer in the following format:\nImproved Answer:"
        )

        response = self.generate_text(prompt)

        if "Improved Answer:" not in response:
            return "[ERROR] No improved answer generated."

        improved_answer = response.rsplit("Improved Answer:", 1)[-1].strip()

        if not improved_answer:
            return "[ERROR] Empty improved answer."

        # Take only the first paragraph
        first_paragraph = improved_answer.split("\n\n")[0].split("\n")[0].strip()
        return clean_bracketed_notes(first_paragraph)

    def generate_self_critique(self, question, answer):
        prompt = (
            "You are an expert evaluator. Write a concise, 50-word critique explaining why the revised answer is an improvement.\n"
            "Focus on clarity, motivation, tone, and detail. Avoid repeating phrases like 'more effective.' Use varied language and be specific.\n\n"
            f"Question:\n{question}\n\nRevised Answer:\n{answer}\n\nCritique (exactly 50 words):"
        )
        response = self.generate_text(prompt, max_new_tokens=100)
        return response.split("Critique (exactly 50 words):", 1)[-1].strip()

    def improve_answer_with_feedback_and_critique(self, question, answer, target_score=1.0):
        # Step 1: Get initial feedback (use dummy values here or hook into self.network)
        original_feedback_dict = self.network.generate_combined_feedback(question, answer)

        expert_feedback = original_feedback_dict.get("expert_feedback", "")
        amateur_feedback = original_feedback_dict.get("amateur_feedback", "")
        initial_score = float(original_feedback_dict.get("combined_score", -1.0))

        if initial_score >= target_score:
            return answer.strip(), initial_score, False  # No improvement needed

        # Step 2: Merge feedback
        condensed_feedback = self.generate_unified_feedback(expert_feedback, amateur_feedback)
        feedback_paragraph = condensed_feedback.strip().split('\n\n')[0].strip()
        print(f"Condensed: {feedback_paragraph}")

        # Step 3: First revision
        improved = self.apply_feedback(question, answer, feedback_paragraph)
        print(f"Improved: {improved}")

        # Step 4: Self-critique the improvement
        critique = self.generate_self_critique(question, improved)

        # Step 5: Refine the improved answer using the critique
        refined = self.apply_feedback(question, improved, critique)
        print(f"Refined: {refined}")

        # Step 6: Final feedback simulation (replace with real feedback if needed)
        final_feedback_dict = self.network.generate_combined_feedback(question, refined, KD=False)
        final_score = float(final_feedback_dict.get("combined_score", -1.0))

        return refined.strip(), final_score, True

In [ ]:
# Define the question first
question = "Why do you want to become a machine learning engineer?"

# Instantiate the model
expert_model = ExpertFeedbackModel()

# Generate the answer
answer = expert_model.generate_answer(question)

# # Improve the answer (assuming this method exists in your class)
# improved_answer, final_score, was_improved = expert_model.improve_answer_with_feedback_and_critique(
#     question, answer, target_score=0.8
# )

# print("--- Result ---")
# print(answer)
# print(improved_answer)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.96 GiB. GPU 0 has a total capacity of 22.16 GiB of which 145.38 MiB is free. Process 4016 has 22.01 GiB memory in use. Of the allocated memory 21.83 GiB is allocated by PyTorch, and 1.25 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print(answer)

In [ ]:
feedback, score = expert_model.generate_feedback(question, answer)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from bert_score import score as bert_score
import re
from difflib import SequenceMatcher


class AmateurExpertFeedbackNetWork:
    def __init__(self, student, teacher, expert_datasets=None, device=None, optimizer=None):
        self.student = student
        self.teacher = teacher
        self.expert_datasets = expert_datasets or []
        self.student_tokenizer = student.tokenizer
        self.teacher_tokenizer = teacher.tokenizer
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.optimizer = optimizer or AdamW(self.student.model.parameters(), lr=5e-5)
        self.align_hidden = None
        self.model_dtype = torch.float32
        self.last_score = 0.0

        for tokenizer in [self.student_tokenizer, self.teacher_tokenizer]:
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

    def knowledge_distillation_with_SFT(self, epochs=3, stopped=None, warmup_alpha=False):
        score_loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            print(f"\n--- Epoch {epoch + 1}/{epochs} ---")
            total_loss_accum = 0

            for i, expert_sample in enumerate(self.expert_datasets):
                if stopped is not None and i == stopped:
                    break

                prompt_text = expert_sample.get("prompt")
                answer = expert_sample.get("answer")
                expert_score = expert_sample.get("score")
                expert_feedback = expert_sample.get("feedback")

                if not prompt_text or not answer or expert_score is None:
                    continue

                print(f"\nSample {i + 1}")

                input_ids, labels, attention_mask = self.student.prepare_inputs_and_labels(
                    prompt_text, answer, expert_score, expert_feedback
                )

                # Forward pass: Student
                student_out = self.student.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    output_hidden_states=True,
                    return_dict=True
                )

                lm_loss = student_out.loss
                student_hidden = student_out.hidden_states[-1][:, 0, :]

                # Feedback & Score from student
                with torch.no_grad():
                    student_feedback, student_score = self.student.generate_feedback(prompt_text, answer)

                # Feedback & Score from teacher
                with torch.no_grad():
                    teacher_feedback, teacher_score = self.teacher.generate_feedback(prompt_text, answer)
                    teacher_inputs = self.teacher_tokenizer(
                        prompt_text, return_tensors="pt", truncation=True, padding=True
                    ).to(self.device)

                    teacher_out = self.teacher.model(
                        input_ids=teacher_inputs["input_ids"],
                        attention_mask=teacher_inputs["attention_mask"],
                        output_hidden_states=True,
                        return_dict=True
                    )
                    teacher_hidden = teacher_out.hidden_states[-1][:, 0, :]

                # Align hidden states
                student_hidden = student_hidden.to(self.model_dtype)
                teacher_hidden = teacher_hidden.to(self.model_dtype)

                if self.align_hidden is None:
                    student_dim = student_hidden.size(-1)
                    teacher_dim = teacher_hidden.size(-1)
                    self.align_hidden = (
                        nn.Linear(student_dim, teacher_dim).to(self.device).to(self.model_dtype)
                        if student_dim != teacher_dim else nn.Identity()
                    )

                student_aligned = self.align_hidden(student_hidden)
                cos_sim = nn.functional.cosine_similarity(student_aligned, teacher_hidden, dim=-1)
                hidden_loss = 1.0 - cos_sim.mean()

                # Score loss (student vs expert)
                student_score_tensor = torch.tensor([student_score], dtype=torch.float32, device=self.device)
                expert_score_tensor = torch.tensor([expert_score], dtype=torch.float32, device=self.device)
                score_loss = score_loss_fn(student_score_tensor, expert_score_tensor)

                # Semantic loss (Student vs Teacher using BERTScore)
                P1, R1, F1 = bert_score([student_feedback], [teacher_feedback], lang="en", verbose=False)
                F1 = F1.to(self.device).to(self.model_dtype)
                semantic_loss_expert = (1.0 - F1).mean()

                # Weighted losses
                alpha = (epoch / epochs * 0.2) if warmup_alpha else 0.2
                beta, gamma, delta = 1.5, 2.0, 0.3

                total_loss = (
                    alpha * lm_loss +
                    beta * score_loss +
                    gamma * semantic_loss_expert +
                    delta * hidden_loss
                )

                self.optimizer.zero_grad()
                total_loss.backward()
                self.optimizer.step()

                total_loss_accum += total_loss.item()

                # Print comparison and metrics
                print("---- FEEDBACK COMPARISON ----")
                print(f"[Student Feedback]: {student_feedback}")
                print(f"[Teacher Feedback]: {teacher_feedback}")
                print(f"[Expert Feedback]: {expert_feedback}")
                print("---- LOSS VALUES ----")
                print(f"LM Loss: {lm_loss.item():.4f}")
                print(f"Score Loss: {score_loss.item():.4f}")
                print(f"Semantic Loss: {semantic_loss_expert.item():.4f}")
                print(f"Hidden Loss: {hidden_loss.item():.4f}")
                print(f"TOTAL LOSS: {total_loss.item():.4f}")
                print("------------------------------\n")

            avg_loss = total_loss_accum / (i + 1) if self.expert_datasets else 0.0
            print(f"\nEpoch {epoch + 1} average loss: {avg_loss:.4f}")

        return self.teacher, self.student


    def generate_combined_feedback(
        self,
        prompt_text,
        answer,
        KD=True,
        max_distilled_attempts=1,
        alpha=0.65,
        distill_threshold=0.90
    ):
        # Step 1: Initial feedback from student and expert
        student_feedback, student_score = self.student.generate_feedback(prompt_text, answer)
        teacher_feedback, teacher_score = self.teacher.generate_feedback(prompt_text, answer)

        try:
            student_score = float(student_score)
        except ValueError:
            student_score = -1.0

        try:
            teacher_score = float(teacher_score)
        except ValueError:
            teacher_score = 1.0  # assume teacher is correct by default

        # Step 2: Semantic similarity check using BERTScore
        _, _, f1 = bert_score([student_feedback], [teacher_feedback], lang="en", verbose=False)
        f1_score = f1[0].item()

        if f1_score < distill_threshold and KD:
            print(f"[Distill Trigger] Similarity too low (F1 = {f1_score:.3f}) — initiating knowledge distillation.")

            for attempt in range(max_distilled_attempts):
                print(f"[Distill Attempt {attempt + 1}] Running knowledge distillation with SFT.")
                self.knowledge_distillation_with_SFT(epochs=3, stopped=20)

                # Step 3: Regenerate feedback after distillation
                updated_student_feedback, updated_student_score = self.student.generate_feedback(prompt_text, answer)
                updated_teacher_feedback, updated_teacher_score = self.teacher.generate_feedback(prompt_text, answer)

                _, _, f1_updated = bert_score([updated_student_feedback], [updated_teacher_feedback], lang="en", verbose=False)
                f1_score_updated = f1_updated[0].item()

                print(f"[Check] Updated F1 (Student vs Teacher): {f1_score_updated:.3f}")

                if f1_score_updated > distill_threshold:
                    print("[Success] Student improved sufficiently — early stopping distillation.")
                    student_feedback = updated_student_feedback
                    teacher_feedback = updated_teacher_feedback
                    try:
                        student_score = float(updated_student_score)
                        teacher_score = float(updated_teacher_score)
                    except ValueError:
                        pass
                    break
            else:
                print("[Warning] Max distillation attempts reached. Student may still underperform.")

        # Step 4: Combine scores
        if (teacher_score > 0 and student_score < 0) or (teacher_score < 0 and student_score > 0):
            # Conflict: favor expert
            combined_score = teacher_score
        else:
            # No major conflict: use weighted average
            combined_score = alpha * teacher_score + (1 - alpha) * student_score

        # Step 5: Combine feedback messages
        combined_feedback = (
            f"Expert says: {teacher_feedback}\n"
            f"Student adds: {student_feedback}"
        )

        print(f"[Combined Feedback Ready] Score: {combined_score:.3f}")
        return {
            "combined_score": combined_score,
            "combined_feedback": combined_feedback,
            "expert_feedback": teacher_feedback,
            "amateur_feedback": student_feedback
        }

In [ ]:
network = AmateurExpertFeedbackNetWork(self_improving_model, expert_model, expert_datasets)
expert_model.network = network

In [ ]:
test_prompt = "Why do you want to become a machine learning engineer?"
answer= self_improving_model.generate_answer(test_prompt)
print(answer)

In [ ]:
answer, _, _ = expert_model.improve_answer_with_feedback_and_critique(test_prompt, answer)

In [ ]:
print(answer)